In [1]:
# Installs missing libraries into the Python that runs this notebook and sets the folders and the main parameters

import importlib, subprocess, sys

for module, package in [("requests", "requests"), ("pandas", "pandas"), ("numpy", "numpy"),
                        ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import hashlib, html, json, re, shutil, time, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Data and model settings
START_MONTH = "2020-01"           # first monthly hiring posts to use
FORECAST_MONTHS = 12              # horizon of the forecasts shown in the app
EXPERIMENT_HORIZONS = [6, 12]     # horizons tested in the forecasting experiment
EXPERIMENT_FROM = "2023-01"       # first date of the forecast experiment
EXPERIMENT_STEP = 3               # months between forecast date
SCENARIO_MIN_GAP_PP = 0.05        # a model "responds to the AI scenario" 
MIN_TRAIN_ORIGINS = 6             # global ML models need at least this many past origins
TIME_MACHINE_CUTOFF = "2023-12"
TRENDS_BY_CUTOFF_FROM = "2022-06" # first month the app's time can go back 
REFRESH_LATEST_MONTH = True       # the newest post keeps growing during its month
FORCE_REBUILD = False             
MIN_MENTIONS = 30                 # a skill with fewer mentions is "limited data"
MIN_POSTS = 50                    # a role with fewer job posts is "limited data"
MIN_WEIGHT = 0.05                 # a skill belongs to a role profile if 5% of the role's posts mention it

# Recommendation rules
ALLOWED_TARGET_EXPOSURE = ["Gradient 2", "Gradient 1", "Minimal Exposure", "Not Exposed"]
TRANSFER_MIN = 0.10               # share of a role's skill weight that admin and finance roles also use

# Demo user 
CV_FILE = "cv01_accounts_executive.txt"   # sample CV to analyse in Steps 10 to 12
CV_TEXT = None                            # paste user's CV text here as a string
TARGET_ROLE = None                        
SDG_PREFERENCE = [8, 12, 13]
SDG_WEIGHT = 0.15

# Folders 
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_HN = ROOT / "data" / "raw" / "hn"
REF = ROOT / "data" / "reference"
CVS = ROOT / "data" / "cvs"
REVIEW = ROOT / "data" / "review"
PROC = ROOT / "data" / "processed"
for folder in (RAW_HN, REF, CVS, REVIEW, PROC):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)
print("Project folder:", ROOT)

Project folder: c:\Users\Asus\Downloads\FutureSkillsRadarV2-main\FutureSkillsRadarV2-main


In [ ]:
# Copy of every reference table

REFERENCE_VERSION = "2.0.4699f257"
OVERWRITE_REFERENCE = False

REFERENCE = {}
REFERENCE["skills.csv"] = r'''
skill_id,skill_name,category,pattern
sql,SQL,data_core,\bsql\b(?!\s+account(ing)?\b)|postgres(ql)?\b|\bmysql\b|\bt-sql\b|\bpl/sql\b
python,Python,programming,\bpython\b
pandas,pandas and dataframes,programming,\bpandas\b|\bpolars\b|\bdataframes?\b
r_lang,R,programming,"\brstudio\b|\btidyverse\b|\bggplot2?\b|\bR programming\b|(?-i:(?<=[\s,(/])R(?=\s*[,/)]))"
excel_advanced,"Excel (pivot tables, lookups, Power Query)",spreadsheets,\b(ms |microsoft )?excel\b(?!\s+(at|in)\b)|pivot ?tables?|\bv ?lookup\b|\bx ?lookup\b|power query
google_sheets,Google Sheets,spreadsheets,google sheets|\bgsheets?\b|google forms
vba_macros,"VBA, macros and Apps Script",spreadsheets,\bvba\b|excel macros?|apps script
power_bi,Power BI,bi_tools,power ?bi\b|\bdax\b
tableau,Tableau,bi_tools,\btableau\b
looker,Looker and Looker Studio,bi_tools,\blooker\b|google data studio
data_visualisation,Data visualisation,data_core,data visuali[sz]ation|\bvisuali[sz]ations?\b|\bd3(\.js)?\b|matplotlib|plotly|seaborn
dashboarding,Dashboards,data_core,\bdashboard(s|ing)?\b
statistics,Statistics,data_core,\bstatistic(s|al)\b|\bregression\b|hypothesis test\w*|\bprobability\b
ab_testing,A/B testing and experimentation,data_core,\ba/b test\w*|\bab test\w*|split test\w*|\bexperimentation\b
data_cleaning,Data cleaning and preparation,data_core,data (cleaning|cleansing|wrangling|preparation|munging)|clean(ing|ed)? (the )?data\b
product_analytics,Product and web analytics,data_core,product analytics|mixpanel|\bamplitude\b|google analytics|\bga4\b|funnel analysis|cohort analysis
data_modelling,Data modelling,data_engineering,data model(l)?ing|dimensional model\w*|star schema
etl_pipelines,ETL and data pipelines,data_engineering,\betl\b|\belt\b|data pipelines?|\bdata ingestion\b
dbt,dbt,data_engineering,\bdbt\b
airflow,Airflow and orchestration,data_engineering,\bairflow\b|\bdagster\b|\bprefect\b
data_warehousing,"Data warehouses (Snowflake, BigQuery)",data_engineering,data ?warehous(e|es|ing)\b|\bsnowflake\b|\bbigquery\b|\bredshift\b|\bdatabricks\b|lakehouse
spark,Spark and big data,data_engineering,\b(py)?spark\b|\bhadoop\b|\bkafka\b
git,Git and version control,engineering,\bgit\b|\bgithub\b|\bgitlab\b|version control
cloud_aws,AWS,cloud,\baws\b|amazon web services
cloud_azure,Microsoft Azure,cloud,\bazure\b
cloud_gcp,Google Cloud,cloud,\bgcp\b|google cloud
api_integration,APIs and integrations,engineering,\bapis?\b|\brest(ful)? apis?\b|\bwebhooks?\b
machine_learning,Machine learning,ai_ml,machine learning|\bml\b|scikit-learn|\bsklearn\b|\bxgboost\b
forecasting,Forecasting and time series,ai_ml,\bforecast(s|ing)?\b|time[- ]series
nlp,Natural language processing,ai_ml,\bnlp\b|natural language processing|text mining
deep_learning,Deep learning,ai_ml,deep learning|neural net\w*|\bpytorch\b|\btensorflow\b|\bkeras\b
mlops,MLOps and LLMOps,ai_ml,\bmlops\b|\bllmops\b|model deployment|model monitoring
llm,Large language models,genai,\bllms?\b|large language models?|foundation models?
generative_ai,Generative AI,genai,generative ai|\bgen ?ai\b
prompt_engineering,Prompt engineering,genai,prompt engineer\w*|prompt design|\bprompting\b
rag,Retrieval augmented generation (RAG),genai,(?-i:\bRAG\b)|retrieval[- ]augmented
vector_databases,Vector databases and embeddings,genai,vector (database|db|store|search)s?|\bembeddings?\b|\bpinecone\b|\bweaviate\b|\bpgvector\b|\bqdrant\b|\bchromadb\b|\bmilvus\b
llm_frameworks,"LLM frameworks (LangChain, LlamaIndex)",genai,\blangchain\b|llama ?index|\blanggraph\b|hugging ?face|semantic kernel
ai_agents,AI agents,genai,\bai agents?\b|\bagentic\b|multi[- ]agent|autonomous agents?
fine_tuning,Fine tuning models,genai,fine[- ]?tun(e|ed|ing)\b|\brlhf\b
ai_assistants,"AI assistants at work (ChatGPT, Copilot, Claude)",genai,\bchatgpt\b|\bcopilot\b|\bgpt[- ]?[345]\w*|\bopenai\b|\banthropic\b|\bclaude\b
responsible_ai,Responsible AI and AI ethics,governance,responsible ai|ai ethics|ethical ai|ai safety|ai governance|algorithmic fairness|bias (testing|mitigation|audit\w*)
workflow_automation,"Workflow automation (Zapier, Make, n8n)",automation,\bzapier\b|\bn8n\b|make\.com|workflow automation|\bintegromat\b
rpa,Robotic process automation (RPA),automation,\brpa\b|robotic process automation|\buipath\b|automation anywhere|blue prism
power_automate,Power Automate and Power Platform,automation,power automate|power apps|power platform|microsoft flow
financial_modelling,Financial modelling,finance,financial model(l)?(ing|s)?\b|\bdcf\b|three[- ]statement model
fpa_budgeting,"Budgeting, FP&A and variance analysis",finance,\bbudgeting\b|budget (planning|management|variance)|\bfp\s?&\s?a\b|financial planning|variance analysis
accounting,"Accounting principles (MFRS, IFRS, GAAP)",finance,\baccounting (principles|standards)\b|\bgaap\b|\bifrs\b|\bmfrs\b|general ledger|journal entr(y|ies)|double[- ]entry
bookkeeping,Bookkeeping,finance,bookkeep(ing|er)
reconciliation,Reconciliation,finance,\breconcil\w*
ap_ar,"Accounts payable and receivable, invoicing",finance,accounts? (payable|receivable)|\bap/ar\b|\bar/ap\b|\binvoic\w*|\bcredit control\b|debt collection
payroll,Payroll,finance,\bpayroll\b|\bepf\b|\bsocso\b|\bpcb\b
financial_reporting,Financial reporting and month end close,finance,financial (reporting|statements?|reports?)|month[- ]end (clos\w*|reporting)|year[- ]end clos\w*|management accounts
audit,Audit and internal control,finance,\baudit(s|ing|ors?)?\b|internal controls?|\bsox\b
tax,Tax and indirect tax (SST),finance,\btax(es|ation)?\b|\bsst\b|\bgst\b|\bvat\b
erp_systems,"ERP systems (SAP, Oracle, NetSuite)",finance,\berp\b|\bsap\b|\bnetsuite\b|oracle (financials|fusion|ebs|erp)|dynamics 365|\bodoo\b
accounting_software,"Accounting software (Xero, QuickBooks, AutoCount, SQL Account)",finance,\bxero\b|\bquickbooks\b|\bautocount\b|sql account(ing)?\b|\bmyob\b|\bsage (50|300|intacct|accounting)\b
einvoicing,E invoicing compliance (MyInvois),finance,\bmyinvois\b|\be-?invoic(e|es|ing)\b
data_entry,Data entry,admin,data entry|data input|keying in|typing speed
document_management,Document and records management,admin,document (management|control)|records? management|filing system|archiv(e|ing) (documents|records)
scheduling_coordination,Scheduling and office coordination,admin,calendar management|diary management|schedul(e|ing) meetings|travel (arrangements|booking)|event coordination|office administration
microsoft_office,Microsoft Office,admin,microsoft office|ms office|office 365|microsoft 365|\bpowerpoint\b|\bms word\b|microsoft word
customer_service,Customer service,admin,customer (service|support)|client service|help ?desk|\bzendesk\b|\bfreshdesk\b
procurement,Procurement and purchasing,admin,\bprocurement\b|purchase orders?|\bpurchasing\b|vendor management
hris,HR information systems,admin,\bhris\b|\bworkday\b|\bbamboohr\b|hr systems?
requirements_gathering,Requirements gathering,business,requirements? (gathering|elicitation|analysis)|user stories|business requirements|functional specifications?
process_improvement,Process improvement,business,process (improvement|optimi[sz]ation|mapping|redesign)|continuous improvement|six sigma|\bbpmn\b|lean (methodolog\w*|principles)
project_management,Project management,business,project management|\bpmp\b|prince2
agile_scrum,Agile and Scrum,business,\bagile\b|\bscrum\b|\bkanban\b|sprint planning
jira,"Jira, Confluence and work trackers",business,\bjira\b|\bconfluence\b|\basana\b|\btrello\b
stakeholder_management,Stakeholder management,business,\bstakeholder\w*
kpi_reporting,KPI and metrics reporting,business,\bkpis?\b|key performance indicators?|\bokrs?\b|metrics reporting|business metrics
crm,"CRM (Salesforce, HubSpot)",business,\bcrm\b|\bsalesforce\b|\bhubspot\b|\bzoho\b
data_governance,Data governance,governance,data governance|data stewardship|master data|data catalog(ue)?|metadata management|data lineage
data_privacy,"Data privacy (PDPA, GDPR)",governance,\bgdpr\b|\bpdpa\b|\bccpa\b|\bhipaa\b|data privacy|data protection
data_quality,Data quality and validation,governance,data quality|data validation|data integrity
esg_reporting,ESG and sustainability reporting,sustainability,\besg\b|sustainability report\w*|\bgri\b|\bissb\b|\btcfd\b|\bcsrd\b
carbon_accounting,Carbon accounting and emissions data,sustainability,carbon (accounting|footprint|emissions?|management|reporting)|greenhouse gas|\bghg\b|scope [123]\b|net[- ]zero|decarboni[sz]\w*
sustainability,Sustainability and climate,sustainability,\bsustainability\b|climate (tech|change|risk|solutions)|renewable energy|clean ?energy|clean ?tech|energy transition
data_storytelling,Data storytelling and presenting insights,soft,data storytelling|storytelling with data|present(ing|ed)? (insights|findings|results)|communicat\w+ (insights|findings)
communication,Communication,soft,communication skills|written and verbal|verbal and written|excellent communicat\w*
problem_solving,Problem solving and analytical thinking,soft,problem[- ]solving|analytical (thinking|skills|mindset)|critical thinking
recruitment,Recruitment and talent acquisition,people_hr,\brecruitment\b|talent acquisition|candidate sourcing|sourcing candidates|\bheadhunt\w*|full[- ]cycle recruit\w*|\bscreening (candidates|cvs|resumes)
employee_onboarding,Employee onboarding,people_hr,(employee|new hire|staff) onboarding|onboarding (of )?new (hires|employees|staff)|new hire orientation|\binduction (programmes?|programs?|training)
employee_relations,Employee relations,people_hr,employee relations|industrial relations|\bgrievances?\b|disciplinary (process|procedures?|hearings?|action)
performance_management,Performance management,people_hr,performance management|performance (appraisals?|reviews?|evaluations?)|appraisal process
employment_law,Employment law (Employment Act),people_hr,employment (act|law|laws|legislation)|labou?r (law|laws|legislation)|industrial relations act|minimum wages? order|wage and hour
compensation_benefits,Compensation and benefits,people_hr,compensation (and|&) benefits|\bc&b\b|benefits administration|salary (benchmarking|structures?)|total rewards
hr_policies,HR policies and procedures,people_hr,\bhr polic(y|ies)|human resources? polic(y|ies)|employee handbook|hr (procedures|sops?)
training_delivery,Training delivery and facilitation,people_hr,training (delivery|sessions?|workshops?|programmes?|programs?)|deliver(ing)? training|facilitat(e|ing|ion of) (workshops?|training|sessions)|train the trainer|(?-i:\bTTT\b)
instructional_design,Instructional design and training needs analysis,people_hr,instructional design|curriculum (design|development)|training needs (analysis|assessment)|(?-i:\bTNA\b)|e-?learning (content|development|modules?)|learning management systems?|(?-i:\bLMS\b)
coaching_mentoring,Coaching and mentoring,people_hr,\bcoaching\b|\bmentor(ing|ship)\b|coach (and|&) mentor
people_management,People management and team leadership,people_hr,people management|team leadership|line management|managing (a )?team|lead(ing)? (a )?team of|supervis(e|ing) (a team|staff|employees)|direct reports
legal_research,Legal research,legal_compliance,legal research|case law|\bstatutes\b|legislative research
legal_documentation,Legal documentation and drafting,legal_compliance,legal (documents?|documentation|drafting)|drafting (contracts|agreements|legal)|contract drafting|\bconveyancing\b|\baffidavits?\b|court (filings?|documents)|litigation support|e-?filing
contract_management,Contract management,legal_compliance,contract (management|administration|review|negotiation)|managing contracts|contract lifecycle
regulatory_compliance,"Regulatory compliance (AML, KYC)",legal_compliance,regulatory compliance|compliance (monitoring|checks?|programmes?|programs?|officer|reviews?|framework)|(?-i:\bAML\b)|anti[- ]money laundering|(?-i:\bKYC\b)|know your customer
corporate_secretarial,"Corporate secretarial (Companies Act, SSM)",legal_compliance,company secretar\w*|corporate secretar\w*|(?-i:\bSSM\b)|companies act|board (minutes|resolutions|meetings)|statutory (filings?|registers?|returns?)|annual returns?
risk_management,Risk management,legal_compliance,risk management|risk registers?|enterprise risk|operational risk|risk (mitigation|controls?)
inventory_management,Inventory and stock control,operations,inventory (management|control|planning|tracking|accuracy)|stock (control|take|taking|management|levels)|\bstocktak\w*|cycle counts?
logistics,Logistics and shipping,operations,\blogistics\b|shipping (and|&) receiving|\bfreight\b|freight forwarding|(?-i:\b3PL\b)|last[- ]mile|\bdispatch(ing)?\b|import (and|&) export|customs clearance
warehouse_management,Warehouse operations,operations,warehous(e|ing) (operations|management|staff)|(?-i:\bWMS\b)|pick(ing)? (and|&) pack(ing)?|order fulfil?ment|distribution cent(re|er)
supply_chain_planning,Supply chain and demand planning,operations,supply chain|demand planning|(?-i:\bS&OP\b)|sales and operations planning|material requirements planning|(?-i:\bMRP\b)
vendor_negotiation,Supplier management and negotiation,operations,supplier (management|negotiations?|relationships?|evaluation|selection|performance)|vendor (negotiations?|selection|evaluation|relationships?)|negotiat\w+ with (vendors|suppliers)|strategic sourcing|sourcing strateg\w*|(?-i:\bRFQs?\b)|(?-i:\bRFPs?\b)|\btender(s|ing)?\b
quality_management,Quality management (ISO 9001),operations,quality management|(?-i:\bQMS\b)|iso\s?9001|quality (control|assurance) (procedures|systems|inspections?)|internal quality audits?
production_planning,Production planning and scheduling,operations,production (planning|scheduling|planner|schedules?)|capacity planning|shop floor|manufacturing operations
event_planning,Event planning and management,operations,event (planning|management|coordination|logistics|operations)|conference (planning|management|logistics)|venue (sourcing|booking|management|selection)|(?-i:\bMICE\b)|trade shows?|exhibitions? (management|logistics)
facilities_management,Facilities and building management,operations,facilit(y|ies) management|building (maintenance|management|operations)|maintenance (contracts|schedules?|scheduling)|preventive maintenance|space planning|office facilities
property_management,Property and tenancy management,operations,property management|tenan(cy|ts?) (management|relations)|lease (management|administration|renewals?)|\bleasing\b|strata management|rent collection
occupational_safety,Occupational safety and health (OSH),safety,occupational (safety|health)|(?-i:\bOSHA?\b)|workplace safety|health (and|&) safety|safety (and|&) health|(?-i:\bDOSH\b)|(?-i:\bHSE\b)|(?-i:\bEHS\b)|(?-i:\bSHE\b)|safety officer
hazard_risk_assessment,Hazard identification and risk assessment (HIRARC),safety,(?-i:\bHIRARC\b)|hazard identification|job safety analysis|(?-i:\bJSA\b)|risk assessments?|(?-i:\bJHA\b)|job hazard analysis
incident_investigation,Incident investigation,safety,incident (investigation|reporting|management)|accident investigation|root cause analysis|near[- ]miss
iso_45001,Safety management systems (ISO 45001),safety,iso\s?45001|ohsas\s?18001|safety management systems?
environmental_compliance,Environmental compliance (ISO 14001),safety,iso\s?14001|environmental (compliance|management systems?|permits?|regulations?|monitoring)|scheduled wastes?|waste management
emergency_response,Emergency response and fire safety,safety,emergency (response|preparedness|evacuation|plans?)|fire safety|fire drills?|first aid
b2b_sales,B2B sales and business development,sales,(?-i:\bB2B\b) sales|business development|new business|sales pipeline|\bprospecting\b|lead generation|cold call\w*|sales (quotas?|targets)|closing deals
account_management,Account management and customer success,sales,account management|key accounts?|client relationship management|customer success|account planning|client retention
negotiation,Negotiation,business,\bnegotiat(e|es|ed|ing|ion|ions|ors?)\b
presentation_skills,Presentation and public speaking,business,presentation skills|public speaking|deliver(ing)? presentations|presenting to (clients|customers|stakeholders|management)
it_support,IT support and troubleshooting,it_support,technical support|tech support|desktop support|(?-i:\bIT support\b)|service desk|end[- ]user support|(first|1st|second|2nd) line support|level [12] support|troubleshoot(ing)? (hardware|software|user|pc|laptop|printer|network)
itsm_ticketing,ITIL and IT service management,it_support,(?-i:\bITIL\b)|(?-i:\bITSM\b)|servicenow|jira service (desk|management)|ticketing (system|tools?)|incident (and|&) problem management
windows_m365_admin,Windows and Microsoft 365 administration,it_support,active directory|windows (server|10|11)|microsoft 365 admin\w*|office 365 admin\w*|\bintune\b|endpoint management|azure ad|entra id
networking_basics,Networking basics,it_support,computer network\w*|network (troubleshooting|administration|support|configuration|fundamentals)|tcp/ip|(?-i:\bDHCP\b)|(?-i:\bVPN\b)|(?-i:\bLAN\b)|(?-i:\bWAN\b)|wi-?fi (setup|troubleshooting)
cybersecurity_awareness,Cybersecurity awareness,it_support,cyber ?security|information security|\bphishing\b|security awareness|(?-i:\bInfoSec\b)
cost_accounting,Cost accounting and costing,finance,cost accounting|\bcosting\b|cost analysis|standard costs?|cost cent(re|er) reporting
attention_to_detail,Attention to detail,soft,attention to detail|detail[- ]oriented|eye for detail
time_management,Time management and prioritisation,soft,time management|prioriti[sz](e|ing|ation)\b|multi-?task\w*|manage multiple (tasks|priorities|deadlines)
teamwork,Teamwork and collaboration,soft,\bteam ?work\b|team player|collaborative (environment|team|mindset)
'''
REFERENCE["roles.csv"] = r'''
role_id,role_name,role_type,title_pattern,isco_code,isco_title,mapping_note,description
paralegal,Paralegal or Legal Assistant,target,\bparalegal|\blegal (assistant|executive|clerk|secretary|associate|administrator)\b|\bconveyancing (clerk|executive|assistant)|\blitigation (assistant|clerk|paralegal)|\bcompliance (assistant|executive|officer|analyst|associate|coordinator),3411,Legal and Related Associate Professionals,Team mapping; compliance officers can also sit in 2422,"Prepares legal documents, research and filings, and supports compliance checks."
hr_executive,HR Executive or HR Generalist,target,\b(hr|human resources?) (executive|generalist|officer|specialist|coordinator|business partner|assistant|associate|administrator)\b|\bpeople (operations|ops|partner)\b|\bpersonnel (officer|executive),2423,Personnel and Careers Professionals,Direct match: Personnel and Careers Professionals. HR Managers (1212) are Minimal exposure; HR clerks (4416) are Gradient 4,"Runs recruitment, onboarding, employee relations and HR policies."
ld_trainer,Training and L&D Executive,target,\btraining (specialist|coordinator|executive|officer|manager|consultant)|\bcorporate trainer|\blearning (and|&) development|\bl&d\b|\binstructional designer|\btraining and development|\bworkshop facilitator|\blearning (specialist|consultant|coordinator),2424,Training and Staff Development Professionals,Direct match: Training and Staff Development Professionals,"Plans and delivers training, from needs analysis to workshops and e learning."
operations_manager,Operations and Administration Manager,target,"\b(business )?operations manager|\badministration manager|\badmin(istrative)? manager|\bgeneral manager,? (operations|administration)|\bhead of (operations|administration)",1219,Business Services and Administration Managers Not Elsewhere Classified,Direct match: Business Services and Administration Managers,"Runs day to day operations: people, processes, vendors and budgets."
event_executive,Event and Conference Executive,target,\bevents? (coordinator|executive|planner|manager|specialist|assistant|producer)|\bconference (coordinator|planner|manager|producer)|\bmeeting (and event )?planner|\bwedding planner,3332,Conference and Event Planners,Direct match: Conference and Event Planners,"Plans and runs events: venues, vendors, budgets and attendees."
business_analyst,Business and Process Analyst,target,business analyst|business systems analyst|operations analyst|\bbizops\b|business operations (analyst|associate)|strategy (and|&) operations|process (analyst|improvement (analyst|specialist)),2421,Management and Organization Analysts,"Team mapping, informed by ESCO",Analyses processes and requirements and recommends improvements.
sales_account_manager,B2B Sales or Account Manager,target,\baccount (manager|executive|director)\b|\bsales (representative|executive|rep|associate|consultant|specialist)\b|\bbusiness development (executive|representative|manager|associate|specialist)|\bb2b sales|\bterritory (sales )?manager|\bkey account|\binside sales,3322,Commercial Sales Representatives,Direct match: Commercial Sales Representatives,"Finds and looks after business customers, from first call to renewal."
it_support,IT Support or Helpdesk Analyst,target,\bit support|\btechnical support (specialist|engineer|analyst|technician|representative)|\bhelp ?desk|\bservice desk|\bdesktop support|\bsupport technician|\bit technician|\bdesktop technician|\bit (helpdesk|service desk) analyst,3512,Information and Communications Technology User Support Technicians,Direct match: ICT User Support Technicians,"Helps colleagues with hardware, software, accounts and network problems."
data_analyst,Data Analyst (comparison),target,\bdata analyst|\banalytics analyst|\binsights? analyst|\bproduct analyst|\bmarketing analyst|\bdata & analytics analyst|\bbi analyst|business intelligence analyst|reporting analyst,2511,Systems Analysts,"Team mapping, informed by ESCO; kept as a comparison with the earlier data focused design","Turns business data into reports, dashboards and answers."
procurement_buyer,Procurement Executive or Buyer,target,\bbuyer\b|\bpurchasing (executive|officer|assistant|specialist|agent|coordinator|manager)|\bprocurement (executive|officer|specialist|analyst|coordinator|assistant|manager)|\bsourcing (specialist|executive|analyst|manager),3323,Buyers,Direct match: Buyers,"Sources and buys goods and services, compares suppliers and negotiates terms."
recruitment_consultant,Recruitment Consultant,target,\brecruit(er|ment consultant|ment specialist|ment coordinator|ment executive|ment officer|ing coordinator|ing specialist)\b|\btalent acquisition (specialist|partner|coordinator|executive|consultant)|\bsourcer\b|\bheadhunter,3333,Employment agents and contractors,Direct match: Employment Agents and Contractors (agency recruiters); in house recruiters can also sit in 2423,"Finds, screens and places candidates for employers."
safety_health_officer,Safety and Health Officer,target,\bsafety (and health )?officer|\bhealth (and|&) safety (officer|specialist|coordinator|advisor|adviser|manager|technician)|\b(ehs|hse|she|osh) (specialist|coordinator|officer|manager|executive|advisor|adviser)|\bsafety (specialist|coordinator|technician|inspector)|\boccupational (health|safety) (specialist|officer|technician),3257,Environmental and Occupational Health Inspectors and Associates,Direct match: Environmental and Occupational Health Inspectors and Associate Professionals,"Keeps workplaces safe: risk assessments, inspections, incident investigation and training."
property_facilities,Property or Facilities Executive,target,\bproperty (manager|executive|administrator|officer|coordinator)|\bfacilit(y|ies) (manager|coordinator|executive|officer|administrator|supervisor)|\bbuilding (manager|executive)|\bleasing (consultant|agent|manager|executive)|\breal estate (agent|associate|executive)|\bstrata,3334,Real Estate Agents and Property Managers,Direct match: Real Estate Agents and Property Managers,"Looks after buildings and tenants: maintenance, leases, vendors and budgets."
senior_accounts,Senior Accounts or Reporting Executive,upgrade,\bsenior accounts? (executive|officer|assistant)|\baccounting (analyst|officer|specialist|associate)|\bgeneral ledger (accountant|specialist)|\bgl accountant|\breporting (accountant|executive)|\bstaff accountant|\bjunior accountant,3313,Accounting Associate Professionals,Team mapping: Accounting Associate Professionals,"Owns month end, reporting and analysis rather than data entry."
office_supervisor,Office or Admin Supervisor,upgrade,\boffice (manager|supervisor|lead)\b|\badmin(istrative)? (supervisor|lead)|\boffice services (supervisor|manager),3341,Office Supervisors,Direct match: Office Supervisors,Leads an admin team and owns office processes and vendors.
accounts_executive,Accounts Executive or Clerk,origin,\baccounts? (executive|assistant|clerk|officer)|\baccount (assistant|clerk)|\baccounting (clerk|assistant)|\baccounts (payable|receivable)|\bap/ar\b|\bbookkeep\w*|\bbilling (specialist|clerk|coordinator),4311,Accounting and Bookkeeping Clerks,Team mapping: Accounting and Bookkeeping Clerks; ISCO places bookkeepers with full responsibility in 3313,"Records transactions, processes invoices and reconciles accounts."
admin_executive,Admin Executive or Office Administrator,origin,\badmin(istrative)? (assistant|executive|officer|clerk|coordinator|associate)|\boffice (assistant|administrator|coordinator|clerk)|\bexecutive assistant|\breceptionist|\bdata entry|\bclerical,4110,General Office Clerks,Team mapping: General Office Clerks; some admin executive roles fit 3343,"Runs office administration, records, scheduling and data entry."
payroll_admin,Payroll Administrator,origin,\bpayroll (clerk|assistant|officer|executive|administrator|specialist|coordinator|analyst),4313,Payroll Clerks,Direct match: Payroll Clerks,"Processes salaries, statutory contributions and payroll records."
customer_service,Customer Service or Contact Centre Officer,origin,\bcustomer (service|support|care) (representative|rep|officer|executive|agent|associate|specialist|advisor|adviser)|\bcall cent(re|er) (agent|representative|executive)|\bcontact cent(re|er) (agent|executive|officer)|\bclient services? (officer|executive|representative),4222,Contact Centre Information Clerks,Direct match: Contact Centre Information Clerks,"Answers customer questions and resolves issues by phone, chat and email."
'''
REFERENCE["ai_terms.csv"] = r'''
term_id,pattern,description
llm,\bllms?\b|large language models?,Large language models
generative_ai,generative ai|\bgen ?ai\b,Generative AI as a named capability
gpt_openai,\bchatgpt\b|\bgpt[- ]?[345]\w*|\bopenai\b,OpenAI models and products
anthropic_claude,\banthropic\b|\bclaude\b,Anthropic models
copilot,\bcopilot\b,GitHub or Microsoft Copilot
open_models,\bllama ?[234]\b|\bmistral\b|\bdeepseek\b|\bqwen\b,Open weight model families
prompt_engineering,prompt engineer\w*,Prompt engineering
rag,(?-i:\bRAG\b)|retrieval[- ]augmented,Retrieval augmented generation
vector_search,vector (database|db|store|search)s?|\bpgvector\b|\bpinecone\b,Vector databases
llm_frameworks,\blangchain\b|llama ?index|\blanggraph\b,LLM application frameworks
ai_agents,\bai agents?\b|\bagentic\b,AI agents
image_generation,stable diffusion|diffusion models?|\bmidjourney\b|\bdall[- ]?e\b,Image generation models
'''
REFERENCE["ilo_exposure.csv"] = r'''
isco_code,occupation,mean_score,sd,gradient,source_page
4132,Data Entry Clerks,0.7,0.03,Gradient 4,48
4131,Typists and Word Processing Operators,0.65,0.05,Gradient 4,48
4311,Accounting and Bookkeeping Clerks,0.64,0.07,Gradient 4,48
4312,"Statistical, Finance and Insurance Clerks",0.64,0.02,Gradient 4,48
3311,Securities and Finance Dealers and Brokers,0.63,0.04,Gradient 4,48
4419,Clerical Support Workers Not Elsewhere Classified,0.63,0.03,Gradient 4,48
2413,Financial Analysts,0.62,0.06,Gradient 4,48
4313,Payroll Clerks,0.61,0.08,Gradient 4,48
5244,Contact Centre Salespersons,0.61,0.1,Gradient 4,48
2513,Web and Multimedia Developers,0.6,0.08,Gradient 4,48
3312,Credit and Loans Officers,0.6,0.04,Gradient 4,48
4110,General Office Clerks,0.6,0.1,Gradient 4,48
4416,Personnel Clerks,0.6,0.09,Gradient 4,48
2643,"Translators, Interpreters and Other Linguists",0.59,0.11,Gradient 3,48
4120,Secretaries (general),0.58,0.1,Gradient 3,48
4211,Bank Tellers and Related Clerks,0.58,0.09,Gradient 3,48
4222,Contact Centre Information Clerks,0.58,0.06,Gradient 3,48
4414,Scribes and Related Workers,0.58,0.08,Gradient 3,48
2412,Financial and Investment Advisers,0.57,0.11,Gradient 3,48
2514,Applications Programmers,0.57,0.08,Gradient 3,48
2521,Database Designers and Administrators,0.57,0.08,Gradient 3,48
2522,Systems Administrators,0.57,0.05,Gradient 3,48
3314,"Statistical, Mathematical and Related Associate Professionals",0.57,0.1,Gradient 3,48
3331,Clearing and Forwarding Agents,0.57,0.09,Gradient 3,48
4225,Inquiry Clerks,0.57,0.09,Gradient 3,48
4226,Receptionists (general),0.57,0.17,Gradient 3,48
2120,"Mathematicians, Actuaries and Statisticians",0.56,0.09,Gradient 3,48
4221,Travel Consultants and Clerks,0.56,0.07,Gradient 3,48
2431,Advertising and Marketing Professionals,0.55,0.09,Gradient 3,48
2519,Software and Applications Developers and Analysts Not Elsewhere Classified,0.55,0.16,Gradient 3,48
2622,Librarians and Related Information Professionals,0.55,0.04,Gradient 3,48
2631,Economists,0.55,0.08,Gradient 3,49
2641,Authors and Related Writers,0.55,0.09,Gradient 3,49
3342,Legal Secretaries,0.55,0.13,Gradient 3,49
4227,Survey and Market Research Interviewers,0.55,0.16,Gradient 3,49
2112,Meteorologists,0.54,0.13,Gradient 3,49
2642,Journalists,0.54,0.1,Gradient 3,49
3343,Administrative and Executive Secretaries,0.54,0.14,Gradient 3,49
4223,Telephone Switchboard Operators,0.54,0.15,Gradient 3,49
2512,Software Developers,0.53,0.07,Gradient 3,49
3321,Insurance Representatives,0.53,0.12,Gradient 3,49
3344,Medical Secretaries,0.53,0.11,Gradient 3,49
3514,Web Technicians,0.53,0.12,Gradient 3,49
2523,Computer Network Professionals,0.52,0.06,Gradient 3,49
3252,Medical Records and Health Information Technicians,0.52,0.1,Gradient 3,49
2411,Accountants,0.51,0.08,Gradient 3,49
2434,Information and Communications Technology Sales Professionals,0.51,0.09,Gradient 3,49
4224,Hotel Receptionists,0.51,0.07,Gradient 3,49
4413,"Coding, Proofreading and Related Clerks",0.51,0.2,Gradient 3,49
2433,Technical and Medical Sales Professionals (excluding ICT),0.5,0.1,Gradient 3,49
4323,Transport Clerks,0.5,0.1,Gradient 3,49
2166,Graphic and Multimedia Designers,0.49,0.09,Gradient 2,49
2511,Systems Analysts,0.49,0.05,Gradient 2,49
2529,Database and Network Professionals Not Elsewhere Classified,0.49,0.06,Gradient 2,49
3313,Accounting Associate Professionals,0.49,0.13,Gradient 2,49
3322,Commercial Sales Representatives,0.49,0.08,Gradient 2,49
3352,Government Tax and Excise Officials,0.49,0.12,Gradient 2,49
2153,Telecommunications Engineers,0.48,0.12,Gradient 2,49
2632,"Sociologists, Anthropologists and Related Professionals",0.48,0.12,Gradient 2,49
3332,Conference and Event Planners,0.48,0.14,Gradient 2,49
4213,Pawnbrokers and Money-lenders,0.48,0.06,Gradient 2,49
1346,Financial and Insurance Services Branch managers,0.47,0.12,Gradient 2,49
2356,Information Technology Trainers,0.47,0.1,Gradient 2,49
2621,Archivists and Curators,0.47,0.11,Gradient 2,49
2633,"Philosophers, Historians and Political Scientists",0.47,0.06,Gradient 2,49
2656,"Announcers on Radio, Television and Other Media",0.47,0.12,Gradient 2,49
3512,Information and Communications Technology User Support Technicians,0.47,0.19,Gradient 2,49
2421,Management and Organization Analysts,0.46,0.1,Gradient 2,49
4411,Library Clerks,0.46,0.18,Gradient 2,49
5243,Door-to-door Salespersons,0.46,0.18,Gradient 2,50
2423,Personnel and Careers Professionals,0.45,0.07,Gradient 2,50
3315,Valuers and Loss Assessors,0.45,0.19,Gradient 2,50
3353,Government Social Benefits Officials,0.45,0.07,Gradient 2,50
4212,"Bookmakers, Croupiers and Related Gaming Workers",0.45,0.19,Gradient 2,50
1330,Information and Communications Technology Service Managers,0.44,0.07,Gradient 2,50
1420,Retail and Wholesale Trade Managers,0.44,0.15,Gradient 2,50
2165,Cartographers and Surveyors,0.44,0.12,Gradient 2,50
3324,Trade Brokers,0.44,0.1,Gradient 2,50
3339,Business Services Agents Not Elsewhere Classified,0.44,0.12,Gradient 2,50
4322,Production Clerks,0.44,0.06,Gradient 2,50
2432,Public Relations Professionals,0.43,0.09,Gradient 2,50
3341,Office Supervisors,0.43,0.11,Gradient 2,50
3354,Government Licensing Officials,0.43,0.07,Gradient 2,50
3511,Information and Communications Technology Operations Technicians,0.43,0.13,Gradient 2,50
3513,Computer Network and Systems Technicians,0.43,0.08,Gradient 2,50
4214,Debt Collectors and Related Workers,0.43,0.07,Gradient 2,50
5221,Shopkeepers,0.43,0.1,Gradient 2,50
1219,Business Services and Administration Managers Not Elsewhere Classified,0.42,0.1,Gradient 2,50
2152,Electronics Engineers,0.42,0.13,Gradient 2,50
4229,Client Information Workers Not Elsewhere Classified,0.42,0.13,Gradient 2,50
5242,Sales Demonstrators,0.42,0.11,Gradient 2,50
1221,Sales and Marketing Managers,0.41,0.09,Gradient 2,50
2424,Training and Staff Development Professionals,0.41,0.09,Gradient 2,50
4412,Mail Carriers and Sorting Clerks,0.41,0.15,Gradient 2,50
2131,"Biologists, Botanists, Zoologists and Related Professionals",0.4,0.1,Gradient 2,50
2351,Education Methods Specialists,0.39,0.11,Gradient 1,50
2634,Psychologists,0.39,0.14,Gradient 1,50
3411,Legal and Related Associate Professionals,0.39,0.14,Gradient 1,50
3431,Photographers,0.39,0.12,Gradient 1,50
5230,Cashiers and Ticket Clerks,0.39,0.14,Gradient 1,50
2111,Physicists and Astronomers,0.38,0.14,Gradient 1,50
3141,Life Science Technicians (excluding Medical),0.38,0.18,Gradient 1,50
5223,Shop Sales Assistants,0.38,0.16,Gradient 1,50
7321,Pre-press Technicians,0.38,0.22,Gradient 1,50
9623,Meter Readers and Vending-machine Collectors,0.38,0.21,Gradient 1,50
1411,Hotel Managers,0.37,0.13,Gradient 1,50
3433,"Gallery, Museum and Library Technicians",0.37,0.2,Gradient 1,51
4415,Filing and Copying Clerks,0.37,0.17,Gradient 1,51
9621,"Messengers, Package Deliverers and Luggage Porters",0.37,0.19,Gradient 1,51
3256,Medical Assistants,0.35,0.23,Gradient 1,51
5211,Stall and Market Salespersons,0.35,0.17,Gradient 1,51
8322,"Car, Taxi and Van Drivers",0.28,0.23,Gradient 1,51
3522,Telecommunications Engineering Technicians,0.45,0.04,Minimal Exposure,51
2422,Policy Administration Professionals,0.42,0.06,Minimal Exposure,51
2164,Town and Traffic Planners,0.41,0.06,Minimal Exposure,51
2265,Dieticians and Nutritionists,0.41,0.07,Minimal Exposure,51
1223,Research and Development Managers,0.4,0.06,Minimal Exposure,51
1222,Advertising and Public Relations Managers,0.39,0.05,Minimal Exposure,51
2113,Chemists,0.39,0.1,Minimal Exposure,51
3323,Buyers,0.39,0.08,Minimal Exposure,51
3333,Employment agents and contractors,0.39,0.09,Minimal Exposure,51
1112,Senior Government Officials,0.38,0.08,Minimal Exposure,51
1120,Managing Directors and Chief Executives,0.38,0.06,Minimal Exposure,51
1321,Manufacturing Managers,0.38,0.11,Minimal Exposure,51
1324,"Supply, Distribution and Related Managers",0.38,0.05,Minimal Exposure,51
1349,Professional Services Managers Not Elsewhere Classified,0.38,0.07,Minimal Exposure,51
2133,Environmental Protection Professionals,0.38,0.04,Minimal Exposure,51
2143,Environmental Engineers,0.38,0.07,Minimal Exposure,51
3114,Electronics Engineering Technicians,0.38,0.11,Minimal Exposure,51
1114,Senior Officials of Special-interest Organizations,0.37,0.06,Minimal Exposure,51
1211,Finance Managers,0.37,0.04,Minimal Exposure,52
1344,Social Welfare Managers,0.37,0.07,Minimal Exposure,52
2141,Industrial and Production Engineers,0.37,0.07,Minimal Exposure,52
2161,Building Architects,0.37,0.08,Minimal Exposure,52
2162,Landscape Architects,0.37,0.08,Minimal Exposure,52
2310,University and Higher Education Teachers,0.37,0.12,Minimal Exposure,52
2619,Legal Professionals Not Elsewhere Classified,0.37,0.09,Minimal Exposure,52
2654,"Film, Stage and Related Directors and Producers",0.37,0.07,Minimal Exposure,52
3422,"Sports Coaches, Instructors and Officials",0.37,0.12,Minimal Exposure,52
3432,Interior Designers and Decorators,0.37,0.08,Minimal Exposure,52
4321,Stock Clerks,0.37,0.06,Minimal Exposure,52
1212,Human Resource Managers,0.36,0.05,Minimal Exposure,52
1342,Health Service Managers,0.36,0.05,Minimal Exposure,52
1345,Education Managers,0.36,0.1,Minimal Exposure,52
1412,Restaurant Managers,0.36,0.12,Minimal Exposure,52
2114,Geologists and geophysicists,0.36,0.1,Minimal Exposure,52
2611,Lawyers,0.36,0.09,Minimal Exposure,52
3359,Government Regulatory Associate Professionals Not Elsewhere Classified,0.36,0.1,Minimal Exposure,52
5222,Shop Supervisors,0.36,0.12,Minimal Exposure,52
1322,Mining Managers,0.35,0.11,Minimal Exposure,52
2145,Chemical Engineers,0.35,0.12,Minimal Exposure,52
2163,Product and Garment Designers,0.35,0.09,Minimal Exposure,53
2320,Vocational Education Teachers,0.35,0.12,Minimal Exposure,53
2354,Other Music Teachers,0.35,0.09,Minimal Exposure,53
2355,Other Arts Teachers,0.35,0.1,Minimal Exposure,53
3118,Draughtspersons,0.35,0.11,Minimal Exposure,53
3122,Manufacturing Supervisors,0.35,0.13,Minimal Exposure,53
3334,Real Estate Agents and Property Managers,0.35,0.09,Minimal Exposure,53
3521,Broadcasting and Audio-visual Technicians,0.35,0.11,Minimal Exposure,53
1323,Construction Managers,0.34,0.06,Minimal Exposure,53
1343,Aged Care Service Managers,0.34,0.09,Minimal Exposure,53
2353,Other Language Teachers,0.34,0.11,Minimal Exposure,53
3155,Air Traffic Safety Electronics Technicians,0.34,0.1,Minimal Exposure,53
2262,Pharmacists,0.33,0.09,Minimal Exposure,53
2359,Teaching Professionals Not Elsewhere Classified,0.33,0.1,Minimal Exposure,53
3412,Social Work Associate Professionals,0.33,0.13,Minimal Exposure,53
1431,"Sports, Recreation and Cultural Centre Managers",0.32,0.1,Minimal Exposure,53
2230,Traditional and Complementary Medicine Professionals,0.32,0.11,Minimal Exposure,53
3351,Customs and Border Inspectors,0.32,0.14,Minimal Exposure,53
5113,Travel Guides,0.32,0.11,Minimal Exposure,53
8219,Assemblers Not Elsewhere Classified,0.32,0.17,Minimal Exposure,53
2655,Actors,0.31,0.15,Minimal Exposure,53
3135,Metal Production Process Controllers,0.31,0.13,Minimal Exposure,54
3154,Air Traffic Controllers,0.31,0.09,Minimal Exposure,54
3212,Medical and Pathology Laboratory Technicians,0.31,0.16,Minimal Exposure,54
7515,Food and Beverage Tasters and Graders,0.31,0.16,Minimal Exposure,54
7543,Product Graders and Testers (except Foods and Beverages),0.31,0.11,Minimal Exposure,54
1341,Child Care Service Managers,0.3,0.15,Minimal Exposure,54
2222,Midwifery Professionals,0.3,0.15,Minimal Exposure,54
3213,Pharmaceutical Technicians and Assistants,0.3,0.12,Minimal Exposure,54
3259,Health Associate Professionals Not Elsewhere Classified,0.3,0.15,Minimal Exposure,54
5161,"Astrologers, Fortune-tellers and Related Workers",0.3,0.16,Minimal Exposure,54
9629,Elementary Workers Not Elsewhere Classified,0.29,0.14,Minimal Exposure,54
2264,Physiotherapists,0.28,0.14,Minimal Exposure,54
5131,Waiters,0.28,0.16,Minimal Exposure,54
8171,Pulp and Papermaking Plant Operators,0.28,0.14,Minimal Exposure,54
8212,Electrical and Electronic Equipment Assemblers,0.28,0.19,Minimal Exposure,54
1311,Agricultural and Forestry Production Managers,0.27,0.13,Minimal Exposure,54
5153,Building Caretakers,0.27,0.16,Minimal Exposure,54
8211,Mechanical Machinery Assemblers,0.27,0.17,Minimal Exposure,54
5112,Transport Conductors,0.25,0.17,Minimal Exposure,54
5322,Home-based Personal Care Workers,0.25,0.18,Minimal Exposure,54
8321,Motorcycle Drivers,0.25,0.17,Minimal Exposure,54
5245,Service Station Attendants,0.24,0.16,Minimal Exposure,55
8332,Heavy Truck and Lorry Drivers,0.24,0.22,Minimal Exposure,55
3221,Nursing Associate Professionals,0.22,0.21,Minimal Exposure,55
1213,Policy and Planning Managers,0.36,0.04,Not Exposed,55
2144,Mechanical Engineers,0.32,0.07,Not Exposed,55
3116,Chemical Engineering Technicians,0.32,0.02,Not Exposed,55
1111,Legislators,0.31,0.04,Not Exposed,55
2151,Electrical Engineers,0.31,0.08,Not Exposed,55
2612,Judges,0.31,0.08,Not Exposed,55
2142,Civil Engineers,0.3,0.06,Not Exposed,55
2149,Engineering Professionals Not Elsewhere Classified,0.3,0.08,Not Exposed,55
2330,Secondary Education Teachers,0.3,0.07,Not Exposed,55
2132,"Farming, Forestry and Fisheries Advisers",0.29,0.1,Not Exposed,55
2146,"Mining Engineers, Metallurgists and Related Professionals",0.29,0.05,Not Exposed,55
2211,Generalist Medical Practitioners,0.29,0.1,Not Exposed,55
3121,Mining Supervisors,0.29,0.08,Not Exposed,55
3133,Chemical Processing Plant Controllers,0.29,0.1,Not Exposed,55
3134,Petroleum and Natural Gas Refining Plant Operators,0.29,0.11,Not Exposed,55
3152,Ships' Deck Officers and Pilots,0.29,0.11,Not Exposed,55
2263,Environmental and Occupational Health and Hygiene Professionals,0.28,0.05,Not Exposed,55
2352,Special Needs Teachers,0.28,0.1,Not Exposed,55
2635,Social Work and Counselling Professionals,0.28,0.1,Not Exposed,55
2652,"Musicians, Singers and Composers",0.28,0.09,Not Exposed,55
3112,Civil Engineering Technicians,0.28,0.07,Not Exposed,55
3117,Mining and metallurgical technicians,0.28,0.06,Not Exposed,55
3123,Construction Supervisors,0.28,0.06,Not Exposed,55
3131,Power Production Plant Operators,0.28,0.11,Not Exposed,55
2212,Specialist Medical Practitioners,0.27,0.12,Not Exposed,55
3113,Electrical Engineering Technicians,0.27,0.05,Not Exposed,55
3132,Incinerator and Water Treatment Plant Operators,0.27,0.09,Not Exposed,55
3153,Aircraft Pilots and Related Associate Professionals,0.27,0.05,Not Exposed,55
8132,Photographic Products Machine Operators,0.27,0.06,Not Exposed,55
2341,Primary School Teachers,0.26,0.09,Not Exposed,55
3111,Chemical and Physical Science Technicians,0.26,0.09,Not Exposed,55
3115,Mechanical Engineering Technicians,0.26,0.08,Not Exposed,55
3119,Physical and Engineering Science Technicians Not Elsewhere Classified,0.26,0.05,Not Exposed,56
3142,Agricultural Technicians,0.26,0.09,Not Exposed,56
5132,Bartenders,0.26,0.12,Not Exposed,56
8121,Metal Processing Plant Operators,0.26,0.1,Not Exposed,56
2221,Nursing Professionals,0.25,0.13,Not Exposed,56
3423,Fitness and Recreation Instructors and Programme Leaders,0.25,0.07,Not Exposed,56
3434,Chefs,0.25,0.06,Not Exposed,56
5241,Fashion and Other Models,0.25,0.06,Not Exposed,56
7322,Printers,0.25,0.11,Not Exposed,56
7421,Electronics Mechanics and Servicers,0.25,0.13,Not Exposed,56
2266,Audiologists and Speech Therapists,0.24,0.04,Not Exposed,56
2267,Optometrists and Ophthalmic Opticians,0.24,0.07,Not Exposed,56
3254,Dispensing Opticians,0.24,0.07,Not Exposed,56
3257,Environmental and Occupational Health Inspectors and Associates,0.24,0.04,Not Exposed,56
5246,Food Service Counter Attendants,0.24,0.13,Not Exposed,56
5312,Teachers' Aides,0.24,0.14,Not Exposed,56
7422,Information and Communications Technology Installers and Servicers,0.24,0.14,Not Exposed,56
8131,Chemical Products Plant and Machine Operators,0.24,0.07,Not Exposed,56
1113,Traditional Chiefs and Heads of Villages,0.23,0.1,Not Exposed,56
2240,Paramedical Practitioners,0.23,0.13,Not Exposed,56
3151,Ships' Engineers,0.23,0.05,Not Exposed,56
3355,Police Inspectors and Detectives,0.23,0.07,Not Exposed,56
3421,Athletes and Sports Players,0.23,0.11,Not Exposed,56
8114,"Cement, Stone and Other Mineral Products Machine Operators",0.23,0.11,Not Exposed,56
1312,Aquaculture and Fisheries Production Managers,0.22,0.08,Not Exposed,56
3211,Medical Imaging and Therapeutic Equipment Technicians,0.22,0.03,Not Exposed,56
3253,Community Health Workers,0.22,0.08,Not Exposed,56
3258,Ambulance Workers,0.22,0.17,Not Exposed,56
5111,Travel Attendants and Travel Stewards,0.22,0.1,Not Exposed,56
5151,"Cleaning and Housekeeping Supervisors in Offices, Hotels and Other Establishments",0.22,0.1,Not Exposed,56
5162,Companions and Valets,0.22,0.05,Not Exposed,56
5212,Street Food Salespersons,0.22,0.11,Not Exposed,56
6123,Apiarists and Sericulturists,0.22,0.11,Not Exposed,56
6221,Aquaculture Workers,0.22,0.15,Not Exposed,56
7323,Print Finishing and Binding Workers,0.22,0.09,Not Exposed,56
7513,Dairy Products Makers,0.22,0.17,Not Exposed,56
7521,Wood Treaters,0.22,0.15,Not Exposed,57
7523,Woodworking Machine Tool Setters and Operators,0.22,0.16,Not Exposed,57
8183,"Packing, Bottling and Labelling Machine Operators",0.22,0.01,Not Exposed,57
2342,Early Childhood Educators,0.21,0.05,Not Exposed,57
2651,Visual Artists,0.21,0.04,Not Exposed,57
3143,Forestry Technicians,0.21,0.07,Not Exposed,57
5169,Personal Services Workers Not Elsewhere Classified,0.21,0.07,Not Exposed,57
7127,Air Conditioning and Refrigeration Mechanics,0.21,0.11,Not Exposed,57
7213,Sheet Metal Workers,0.21,0.09,Not Exposed,57
7311,Precision-instrument Makers and Repairers,0.21,0.08,Not Exposed,57
8112,Mineral and Stone Processing Plant Operators,0.21,0.1,Not Exposed,57
8154,"Bleaching, Dyeing and Fabric Cleaning Machine Operators",0.21,0.15,Not Exposed,57
9331,Hand and Pedal Vehicle Drivers,0.21,0.08,Not Exposed,57
2269,Health Professionals Not Elsewhere Classified,0.2,0.08,Not Exposed,57
3222,Midwifery Associate Professionals,0.2,0.07,Not Exposed,57
5152,Domestic Housekeepers,0.2,0.1,Not Exposed,57
5165,Driving Instructors,0.2,0.03,Not Exposed,57
5414,Security Guards,0.2,0.09,Not Exposed,57
5419,Protective Services Workers Not Elsewhere Classified,0.2,0.05,Not Exposed,57
6129,Animal Producers Not Elsewhere Classified,0.2,0.14,Not Exposed,57
7222,Toolmakers and Related Workers,0.2,0.1,Not Exposed,57
8122,"Metal Finishing, Plating and Coating Machine Operators",0.2,0.04,Not Exposed,57
8182,Steam Engine and Boiler Operators,0.2,0.04,Not Exposed,57
8311,Locomotive Engine Drivers,0.2,0.08,Not Exposed,57
8344,Lifting Truck Operators,0.2,0.1,Not Exposed,57
9321,Hand Packers,0.2,0.05,Not Exposed,57
9334,Shelf Fillers,0.2,0.1,Not Exposed,57
9520,Street Vendors (excluding Food),0.2,0.1,Not Exposed,57
2653,Dancers and Choreographers,0.19,0.1,Not Exposed,57
3230,Traditional and Complementary Medicine Associate Professionals,0.19,0.11,Not Exposed,57
5311,Child Care Workers,0.19,0.09,Not Exposed,57
6122,Poultry Producers,0.19,0.1,Not Exposed,57
6130,Mixed Crop and Animal Producers,0.19,0.11,Not Exposed,57
7232,Aircraft Engine Mechanics and Repairers,0.19,0.07,Not Exposed,57
7315,"Glass Makers, Cutters, Grinders and Finishers",0.19,0.06,Not Exposed,57
7411,Building and Related Electricians,0.19,0.06,Not Exposed,57
8113,Well Drillers and Borers and Related Workers,0.19,0.12,Not Exposed,57
3214,Medical and Dental Prosthetic Technicians,0.18,0.05,Not Exposed,57
5120,Cooks,0.18,0.06,Not Exposed,57
5142,Beauticians and Related Workers,0.18,0.19,Not Exposed,58
5411,Fire Fighters,0.18,0.1,Not Exposed,58
6111,Field Crop and Vegetable Growers,0.18,0.12,Not Exposed,58
6113,"Gardeners, Horticultural and Nursery Growers",0.18,0.11,Not Exposed,58
6223,Deep-sea Fishery Workers,0.18,0.07,Not Exposed,58
7126,Plumbers and Pipe Fitters,0.18,0.1,Not Exposed,58
7223,Metal Working Machine Tool Setters and Operators,0.18,0.05,Not Exposed,58
7231,Motor Vehicle Mechanics and Repairers,0.18,0.04,Not Exposed,58
7313,Jewellery and Precious Metal Workers,0.18,0.05,Not Exposed,58
7314,Potters and Related Workers,0.18,0.07,Not Exposed,58
7316,"Sign Writers, Decorative Painters, Engravers and Etchers",0.18,0.08,Not Exposed,58
7534,Upholsterers and Related Workers,0.18,0.09,Not Exposed,58
8141,Rubber Products Machine Operators,0.18,0.05,Not Exposed,58
8143,Paper Products Machine Operators,0.18,0.02,Not Exposed,58
8331,Bus and Tram Drivers,0.18,0.06,Not Exposed,58
8343,"Crane, hoist and related plant operators",0.18,0.03,Not Exposed,58
9411,Fast Food Preparers,0.18,0.05,Not Exposed,58
9510,Street and Related Service Workers,0.18,0.09,Not Exposed,58
9612,Refuse Sorters,0.18,0.11,Not Exposed,58
2636,Religious Professionals,0.17,0.12,Not Exposed,58
3255,Physiotherapy Technicians and Assistants,0.17,0.07,Not Exposed,58
5141,Hairdressers,0.17,0.18,Not Exposed,58
5163,Undertakers and Embalmers,0.17,0.17,Not Exposed,58
6112,Tree and Shrub Crop Growers,0.17,0.11,Not Exposed,58
6114,Mixed Crop Growers,0.17,0.12,Not Exposed,58
6121,Livestock and Dairy Producers,0.17,0.1,Not Exposed,58
6222,Inland and Coastal Waters Fishery Workers,0.17,0.1,Not Exposed,58
7221,"Blacksmiths, Hammersmiths and Forging Press Workers",0.17,0.05,Not Exposed,58
7224,"Metal Polishers, Wheel Grinders and Tool Sharpeners",0.17,0.05,Not Exposed,58
7233,Agricultural and Industrial Machinery Mechanics and Repairers,0.17,0.04,Not Exposed,58
7412,Electrical Mechanics and Fitters,0.17,0.05,Not Exposed,58
7512,"Bakers, Pastry-cooks and Confectionery Makers",0.17,0.08,Not Exposed,58
7532,Garment and Related Patternmakers and Cutters,0.17,0.04,Not Exposed,58
7536,Shoemakers and Related Workers,0.17,0.06,Not Exposed,58
8111,Miners and Quarriers,0.17,0.11,Not Exposed,58
8142,Plastic Products Machine Operators,0.17,0.05,Not Exposed,58
8181,Glass and Ceramics Plant Operators,0.17,0.06,Not Exposed,58
8312,"Railway Brake, Signal and Switch Operators",0.17,0.04,Not Exposed,58
3251,Dental Assistants and Therapists,0.16,0.1,Not Exposed,58
7522,Cabinet-makers and Related Workers,0.16,0.06,Not Exposed,59
7541,Underwater Divers,0.16,0.09,Not Exposed,59
8152,Weaving and Knitting Machine Operators,0.16,0.03,Not Exposed,59
8156,Shoemaking and Related Machine Operators,0.16,0.02,Not Exposed,59
8157,Laundry Machine Operators,0.16,0.03,Not Exposed,59
8159,"Textile, Fur and Leather Products Machine Operators Not Elsewhere Classified",0.16,0.01,Not Exposed,59
2261,Dentists,0.15,0.06,Not Exposed,59
5329,Personal Care Workers in Health Services Not Elsewhere Classified,0.15,0.03,Not Exposed,59
7413,Electrical Line Installers and Repairers,0.15,0.08,Not Exposed,59
7514,"Fruit, Vegetable and Related Preservers",0.15,0.04,Not Exposed,59
7531,"Tailors, Dressmakers, Furriers and Hatters",0.15,0.05,Not Exposed,59
8151,"Fibre Preparing, Spinning and Winding Machine Operators",0.15,0.02,Not Exposed,59
8153,Sewing Machine Operators,0.15,0.03,Not Exposed,59
8155,Fur and Leather Preparing Machine Operators,0.15,0.02,Not Exposed,59
8160,Food and Related Products Machine Operators,0.15,0.03,Not Exposed,59
2250,Veterinarians,0.14,0.05,Not Exposed,59
3240,Veterinary Technicians and Assistants,0.14,0.02,Not Exposed,59
5164,Pet Groomers and Animal Care Workers,0.14,0.06,Not Exposed,59
5321,Health Care Assistants,0.14,0.06,Not Exposed,59
5412,Police Officers,0.14,0.03,Not Exposed,59
5413,Prison Guards,0.14,0.04,Not Exposed,59
7111,House Builders,0.14,0.07,Not Exposed,59
7125,Glaziers,0.14,0.0,Not Exposed,59
7312,Musical Instrument Makers and Tuners,0.14,0.02,Not Exposed,59
7317,"Handicraft Workers in Wood, Basketry and Related Materials",0.14,0.01,Not Exposed,59
7516,Tobacco Preparers and Tobacco Products Makers,0.14,0.06,Not Exposed,59
7549,Craft and Related Workers not Elsewhere Classified,0.14,0.01,Not Exposed,59
8172,Wood Processing Plant Operators,0.14,0.05,Not Exposed,59
8350,Ships' Deck Crews and Related Workers,0.14,0.02,Not Exposed,59
9111,Domestic Cleaners and Helpers,0.14,0.08,Not Exposed,59
9121,Hand Launderers and Pressers,0.14,0.03,Not Exposed,59
9333,Freight Handlers,0.14,0.03,Not Exposed,59
2659,Creative and Performing Artists Not Elsewhere Classified,0.13,0.06,Not Exposed,59
6310,Subsistence Crop Farmers,0.13,0.05,Not Exposed,59
6320,Subsistence Livestock Farmers,0.13,0.04,Not Exposed,59
7115,Carpenters and Joiners,0.13,0.02,Not Exposed,59
7121,Roofers,0.13,0.08,Not Exposed,59
7124,Insulation Workers,0.13,0.02,Not Exposed,60
7131,Painters and Related Workers,0.13,0.04,Not Exposed,60
7211,Metal Moulders and Coremakers,0.13,0.01,Not Exposed,60
7212,Welders and Flame Cutters,0.13,0.05,Not Exposed,60
7215,Riggers and Cable Splicers,0.13,0.03,Not Exposed,60
7234,Bicycle and Related Repairers,0.13,0.02,Not Exposed,60
7318,"Handicraft Workers in Textile, Leather and Related Materials",0.13,0.03,Not Exposed,60
7511,"Butchers, Fishmongers and Related Food Preparers",0.13,0.03,Not Exposed,60
7544,Fumigators and Other Pest and Weed Controllers,0.13,0.04,Not Exposed,60
8342,Earthmoving and Related Plant Operators,0.13,0.02,Not Exposed,60
9332,Drivers of Animal-drawn Vehicles and Machinery,0.13,0.08,Not Exposed,60
9412,Kitchen Helpers,0.13,0.02,Not Exposed,60
3413,Religious Associate Professionals,0.12,0.07,Not Exposed,60
6210,Forestry and Related Workers,0.12,0.05,Not Exposed,60
6330,Subsistence Mixed Crop and Livestock Farmers,0.12,0.05,Not Exposed,60
6340,"Subsistence Fishers, Hunters, Trappers and Gatherers",0.12,0.05,Not Exposed,60
7132,Spray Painters and Varnishers,0.12,0.0,Not Exposed,60
7533,"Sewing, Embroidery and Related Workers",0.12,0.02,Not Exposed,60
7542,Shotfirers and Blasters,0.12,0.05,Not Exposed,60
8341,Mobile Farm and Forestry Plant Operators,0.12,0.03,Not Exposed,60
9112,"Cleaners and Helpers in Offices, Hotels and Other Establishments",0.12,0.02,Not Exposed,60
9212,Livestock Farm Labourers,0.12,0.03,Not Exposed,60
9214,Garden and Horticultural Labourers,0.12,0.02,Not Exposed,60
9329,Manufacturing Labourers Not Elsewhere Classified,0.12,0.02,Not Exposed,60
7113,"Stonemasons, Stone Cutters, Splitters and Carvers",0.11,0.02,Not Exposed,60
7123,Plasterers,0.11,0.01,Not Exposed,60
7214,Structural Metal Preparers and Erectors,0.11,0.02,Not Exposed,60
7535,"Pelt Dressers, Tanners and Fellmongers",0.11,0.03,Not Exposed,60
9123,Window Cleaners,0.11,0.03,Not Exposed,60
9213,Mixed Crop and Livestock Farm Labourers,0.11,0.03,Not Exposed,60
9216,Fishery and Aquaculture Labourers,0.11,0.03,Not Exposed,60
9311,Mining and Quarrying Labourers,0.11,0.02,Not Exposed,60
9622,Odd Job Persons,0.11,0.03,Not Exposed,60
7114,"Concrete Placers, Concrete Finishers and Related Workers",0.1,0.03,Not Exposed,60
7122,Floor Layers and Tile Setters,0.1,0.02,Not Exposed,60
9129,Other Cleaning Workers,0.1,0.01,Not Exposed,60
6224,Hunters and Trappers,0.09,0.03,Not Exposed,60
7112,Bricklayers and Related Workers,0.09,0.01,Not Exposed,60
7119,Building Frame and Related Trades Workers Not Elsewhere Classified,0.09,0.01,Not Exposed,61
7133,Building Structure Cleaners,0.09,0.01,Not Exposed,61
9122,Vehicle Cleaners,0.09,0.01,Not Exposed,61
9211,Crop Farm Labourers,0.09,0.02,Not Exposed,61
9215,Forestry Labourers,0.09,0.02,Not Exposed,61
9312,Civil Engineering Labourers,0.09,0.02,Not Exposed,61
9313,Building Construction Labourers,0.09,0.02,Not Exposed,61
9611,Garbage and Recycling Collectors,0.09,0.02,Not Exposed,61
9613,Sweepers and Related Labourers,0.09,0.03,Not Exposed,61
9624,Water and Firewood Collectors,0.09,0.03,Not Exposed,61
'''
REFERENCE["courses.csv"] = r'''
course_id,title,provider,url,skills,level,hours,cost_rm,funding_route,language,checked_on,notes
C01,Intro to SQL,Kaggle,https://www.kaggle.com/learn/intro-to-sql,sql,beginner,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description. Uses Google BigQuery.
C02,Advanced SQL,Kaggle,https://www.kaggle.com/learn/advanced-sql,sql,intermediate,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description. Description: 'Take your SQL skills to the next level'.
C03,Python,Kaggle,https://www.kaggle.com/learn/python,python,beginner,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description.
C04,Pandas,Kaggle,https://www.kaggle.com/learn/pandas,pandas,beginner,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description. Short hands on data manipulation challenges.
C05,Data Cleaning,Kaggle,https://www.kaggle.com/learn/data-cleaning,data_cleaning,intermediate,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description.
C06,Intro to Machine Learning,Kaggle,https://www.kaggle.com/learn/intro-to-machine-learning,machine_learning,beginner,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description.
C07,Time Series,Kaggle,https://www.kaggle.com/learn/time-series,forecasting;machine_learning,intermediate,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description. Description: 'Apply machine learning to real-world forecasting tasks'.
C08,Intro to AI Ethics,Kaggle,https://www.kaggle.com/learn/intro-to-ai-ethics,responsible_ai,beginner,,0.0,Free,English,2026-10-22,Free per Kaggle Learn catalogue page ('no-cost courses'). No duration or provider level shown; level judged from description.
C09,Get started building with Power BI,Microsoft Learn,https://learn.microsoft.com/en-us/training/modules/get-started-with-power-bi/,power_bi;data_visualisation,beginner,,0.0,Free,English,2026-10-22,Provider label Beginner; no prerequisites; 6 units incl. Power BI service exercise. Free per Microsoft Learn FAQ. No duration shown on page.
C10,Model data with Power BI,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/model-data-power-bi/,power_bi;data_modelling,intermediate,,0.0,Free,English,2026-10-22,Provider label Intermediate; 6 modules on semantic models and DAX; assumes familiarity with Power BI Desktop. Free per Microsoft Learn FAQ. No duration shown on page.
C11,Design effective reports in Power BI,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/power-bi-effective/,power_bi;data_visualisation;data_storytelling,intermediate,,0.0,Free,English,2026-10-22,Provider label Intermediate; overview cites 'data visualization and storytelling techniques'. Free per Microsoft Learn FAQ. No duration shown on page.
C12,Automate a business process using Power Automate,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/automate-process-power-automate/,power_automate;workflow_automation,beginner,,0.0,Free,English,2026-10-22,Provider label Beginner; 7 modules incl. approval flows and AI Builder. Free per Microsoft Learn FAQ. No duration shown on page.
C13,Introduction to generative AI and agents,Microsoft Learn,https://learn.microsoft.com/en-us/training/modules/fundamentals-generative-ai/,generative_ai;llm;prompt_engineering;ai_agents,beginner,,0.0,Free,English,2026-10-22,"Provider label Beginner; covers LLMs, prompts and agents; prerequisites: familiarity with AI concepts. Free per Microsoft Learn FAQ. No duration shown on page. FAQ notes some exercises need an Azure subscription. Also tagged ai_agents: Title and module cover AI agents."
C14,Introduction to Microsoft Purview and Microsoft's privacy principles,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/describe-capabilities-of-microsoft-compliance-solutions/,data_privacy;data_governance,beginner,,0.0,Free,English,2026-10-22,Provider label Beginner; 4 modules incl. privacy principles and Purview data governance. Free per Microsoft Learn FAQ. No duration shown on page.
C15,Introduction to Generative AI,Google Cloud Skills Boost,https://www.skills.google/course_templates/536,generative_ai,beginner,0.75,0.0,Free,English,2026-10-22,Page states 45 minutes and Introductory. Described as 'no-cost' in Google Cloud training blog; old cloudskillsboost.google link redirects here.
C16,ChatGPT Prompt Engineering for Developers,DeepLearning.AI,https://www.deeplearning.ai/courses/chatgpt-prompt-eng,prompt_engineering;llm;generative_ai,beginner,1.67,0.0,Free with certificate fee,English,2026-10-22,Provider label Beginner; page states 1h40m. Access 'free for a limited time' during platform beta; accomplishment needs PRO membership (price not shown). Basic Python needed.
C17,Learn Data Analysis with Python in this Comprehensive 19-Hour Bootcamp,freeCodeCamp,https://www.freecodecamp.org/news/learn-data-analysis-with-comprehensive-19-hour-bootcamp/,sql;excel_advanced;tableau;power_bi;python;pandas;data_visualisation,beginner,19.0,0.0,Free,English,2026-10-22,"No level stated; judged beginner. YouTube video; freeCodeCamp says its videos are 'all freely available to the public'. Covers SQL, Excel pivot tables and XLOOKUP, Tableau, Power BI with DAX, Python and Pandas. Published 9 January 2024."
C18,"Learn Python for Data Science – Hands-on Projects with EDA, AB Testing & Business Intelligence",freeCodeCamp,https://www.freecodecamp.org/news/learn-python-for-data-science-hands-on-projects-with-eda-ab-testing-business-intelligence/,ab_testing;statistics;python;data_visualisation,intermediate,5.5,0.0,Free,English,2026-10-22,No level stated; judged intermediate. Page states '5.5+ hour' course on freeCodeCamp YouTube (videos 'all freely available to the public'). Covers A/B testing and descriptive statistics. Published 11 June 2024.
C19,Exploring data: Graphs and numerical summaries,The Open University (OpenLearn),https://www.open.edu/openlearn/science-maths-technology/mathematics-statistics/exploring-data-graphs-and-numerical-summaries/content-section-0,statistics;data_visualisation,beginner,20.0,0.0,Free,English,2026-10-22,Free course; Level 1: Introductory; 20 hours study; free statement of participation.
C20,Data analysis: hypothesis testing,The Open University (OpenLearn),https://www.open.edu/openlearn/science-maths-technology/data-analysis-hypothesis-testing/content-section-0?intro=1,statistics,beginner,9.0,0.0,Free,English,2026-10-22,Free course; Level 1: Introductory; 9 hours study; requires Microsoft Excel or similar; free statement of participation or badge.
C21,dbt Fundamentals,dbt Labs (dbt Learn),https://learn.getdbt.com/courses/dbt-fundamentals,dbt,beginner,5.0,0.0,Free,English,2026-10-22,"Course page needs JavaScript; title, 5 hours and level 'New to dbt' verified on getdbt.com/dbt-learn, which says 'Start learning free'."
C22,Learn Tableau Desktop,Tableau,https://www.tableau.com/learn/training,tableau;data_visualisation,beginner,0.43,0.0,Free,English,2026-10-22,"Series on Tableau 'Free Training Videos' page: 8 videos, 26 min. No level stated; judged beginner (getting started series)."
C23,Powerful data analysis and data modeling in excel,HRD Corp (e-LATiH),https://elatih.hrdcorp.gov.my/courses-details/powerful-data-analysis-and-data-modeling-in-excel,excel_advanced;data_modelling;etl_pipelines;data_cleaning,intermediate,0.83,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-22,"Page states 50 Min; no level stated, judged intermediate (Power Query and Power Pivot). HRD Corp certificate on completion. Trainer Shabenah Bee Binti Mohd. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
C24,Corporate Sustainability Fundamentals,HRD Corp (e-LATiH),https://elatih.hrdcorp.gov.my/courses-details/corporate-sustainability-fundamentals,sustainability;esg_reporting,beginner,,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-22,"No duration, level or trainer stated; judged beginner (no prerequisites). Covers GRI and measuring and reporting sustainability performance. Some e-LATiH pages failed to load today. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
C25,Introduction to Sustainable Finance,UN CC:e-Learn,https://unccelearn.org/course/view.php?id=139&page=overview&lang=en,sustainability,beginner,10.0,0.0,Free with certificate fee,English,2026-10-22,"Self paced; page states 10 hours; no level stated, judged beginner. No enrolment fee shown; UN certificate released after payment of USD 20 (quizzes 70% pass)."
A01,Human resources: recruitment and selection,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/human-resources/human-resources-recruitment-and-selection/content-section-0?intro=1,recruitment,intermediate,12.0,0.0,Free,English,2026-10-24,OpenLearn label Level 2: Intermediate; page states 12 hours study; free statement of participation; free account needed to track progress
A02,Managing Recruitment,Learning Pool via Kaya (Humanitarian Leadership Academy),https://kayaconnect.org/course/info.php?id=415,recruitment;people_management,beginner,1.0,0.0,Free,English,2026-10-24,"Page says about 60 minutes and This course is free; level not stated, beginner is my judgement (aimed at managers of any level); covers HR planning, legal framework, job analysis, advertising and selection"
A03,Onboarding Principles for Employees,Alison,https://alison.com/course/onboarding-principles-for-employees,employee_onboarding,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 3 to 4 hours; the upper figure is used; Alison label Intermediate; study free, certificate fee applies (amount not shown on page)"
A04,Human Resource Management - Discipline in Organizations,Alison,https://alison.com/course/human-resource-management-discipline-in-organizations-revised,employee_relations;employment_law;hr_policies,beginner,,0.0,Free with certificate fee,English,2026-10-24,"No duration shown on page; Alison label Beginner; covers just cause, insubordination and workplace bullying; study free, certificate fee applies (amount not shown)"
A05,Performance Appraisal in HR: A Strategic Approach to Performance Management,Alison,https://alison.com/course/performance-appraisal-in-hr-a-strategic-approach-to-performance-management,performance_management,intermediate,6.0,0.0,Free with certificate fee,English,2026-10-24,Page states 5 to 6 hours; the upper figure is used; Alison label Intermediate; certificate fee applies (amount not shown)
A06,Managing individuals: managing performance,UNHCR via Kaya (Humanitarian Leadership Academy),https://kayaconnect.org/course/info.php?id=378,performance_management;people_management,beginner,0.5,0.0,Free,English,2026-10-24,"Page says about 30 minutes and This course is free; covers setting objectives and delivering appraisals; level not stated, beginner is my judgement"
A07,Introduction to International Labour Standards,ILO International Training Centre (ITCILO),https://www.itcilo.org/courses/introduction-international-labour-standards,employment_law;regulatory_compliance,beginner,2.0,0.0,Free,English,2026-10-24,"Self guided; page says set aside around two hours; also in French, Spanish and Portuguese; international standards, not Malaysian law; level not stated, beginner is my judgement (non specialist audience)"
A08,Professional in Compensation and Benefits,Alison,https://alison.com/course/professional-in-compensation-and-benefits,compensation_benefits,intermediate,10.0,0.0,Free with certificate fee,English,2026-10-24,Page states 6 to 10 hours; the upper figure is used; Alison label Intermediate; certificate fee applies (amount not shown)
A09,Human Resource Policies and Organization Structure,Alison,https://alison.com/course/human-resource-policies-and-organization-structure,hr_policies,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Advanced; certificate fee applies (amount not shown)
A10,Introduction to Training of Trainers,Alison,https://alison.com/course/introduction-to-training-of-trainers,training_delivery;presentation_skills,beginner,5.0,0.0,Free with certificate fee,English,2026-10-24,Page states 4 to 5 hours; the upper figure is used; Alison label Beginner; covers running training programmes in organisations; certificate fee applies (amount not shown)
A11,The Essentials of Educating Adults,Alison,https://alison.com/course/the-essentials-of-educating-adults,training_delivery;instructional_design,advanced,5.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 4 to 5 hours; the upper figure is used; Alison label Advanced; covers trainer roles, pedagogical principles and session design; certificate fee applies (amount not shown)"
A12,Instructional Design Principles to Create Successful E-Learning Courses,Alison,https://alison.com/course/instructional-design-principles-to-create-successful-e-learning-courses,instructional_design,advanced,5.0,0.0,Free with certificate fee,English,2026-10-24,Page states 4 to 5 hours; the upper figure is used; Alison label Advanced; certificate fee applies (amount not shown)
A13,Take your teaching online,The Open University (OpenLearn),https://www.open.edu/openlearn/education-development/education/take-your-teaching-online,instructional_design;training_delivery,intermediate,24.0,0.0,Free,English,2026-10-24,"OpenLearn label Level 2: Intermediate; page states 24 hours study; 24 CPD points; free statement of participation; covers effective, inclusive and accessible online teaching"
A14,Workplace learning with coaching and mentoring,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/workplace-learning-coaching-and-mentoring/content-section-0?intro=1,coaching_mentoring;training_delivery,advanced,9.0,0.0,Free,English,2026-10-24,OpenLearn label Level 3: Advanced; page states 9 hours study; covers planning learning and talent development plus coaching and mentoring; free statement of participation
A15,Managing and managing people,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/leadership-management/managing-and-managing-people/content-section-0,people_management,advanced,8.0,0.0,Free,English,2026-10-24,OpenLearn label Level 3: Advanced; page states 8 hours study; free statement of participation
A16,Introducing research in law and beyond,The Open University (OpenLearn),https://www.open.edu/openlearn/society-politics-law/introducing-research-law-and-beyond/content-section-0?intro=1,legal_research,advanced,10.0,0.0,Free,English,2026-10-24,"OpenLearn label Level 3: Advanced; page states 10 hours study; covers planning a legal research project, literature reviews, methodology and ethics; UK academic context"
A17,Legal Administrative Assistant Essentials,Alison,https://alison.com/course/legal-administrative-assistant-essentials,legal_documentation;document_management,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Intermediate; covers preparing legal documents and managing case files; certificate fee applies (amount not shown)
A18,Legal Office Administration,Alison,https://alison.com/course/legal-office-administration,legal_documentation;contract_management,beginner,6.0,0.0,Free with certificate fee,English,2026-10-24,Page states 5 to 6 hours; the upper figure is used; Alison label Beginner; topics include preparing legal documents and contract language; jurisdiction not stated; certificate fee applies (amount not shown)
A19,Essentials of Contract Management,Alison,https://alison.com/course/essentials-of-contract-management,contract_management,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 3 to 4 hours; the upper figure is used; Alison label Intermediate; covers negotiation, drafting, execution and management; CPD accredited; certificate fee applies (amount not shown)"
A20,Basics of Contract Law,Alison,https://alison.com/courses/basics-of-contracts-law/content,contract_management;legal_documentation,advanced,4.0,0.0,Free with certificate fee,English,2026-10-24,Page states 3 to 4 hours; the upper figure is used; Alison label Advanced; certificate fee applies (amount not shown)
A21,Basics of Compliance Management,Alison,https://alison.com/course/basics-of-compliance-management,regulatory_compliance;risk_management,advanced,4.0,0.0,Free with certificate fee,English,2026-10-24,Page states 3 to 4 hours; the upper figure is used; Alison label Advanced; covers compliance management and risk detection systems; certificate fee applies (amount not shown)
A22,Company Law in Context,Alison,https://alison.com/course/company-law-in-context,corporate_secretarial;legal_documentation,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Intermediate; covers incorporation paperwork and governance; does not cover company secretary duties or Malaysian Companies Act 2016; certificate fee applies (amount not shown)
A23,Corporate Governance,Alison,https://alison.com/course/corporate-governance,corporate_secretarial;regulatory_compliance,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 1.5 to 3 hours; the upper figure is used; Alison label Beginner; covers roles of owners, board and management; company secretary role not covered; certificate fee applies (amount not shown)"
A24,Minute-Taking Techniques and Best Practices,Alison,https://alison.com/course/minute-taking-techniques-and-best-practices,corporate_secretarial;document_management;communication,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Beginner; certificate fee applies (amount not shown)
A25,Understanding the General Data Protection Regulation (GDPR),Alison,https://alison.com/course/understanding-the-general-data-protection-regulation-gdpr,data_privacy;regulatory_compliance,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 1.5 to 3 hours; the upper figure is used; Alison label Beginner; EU law, useful background for PDPA but not Malaysian law; certificate fee applies (amount not shown)"
A26,Malaysia's Personal Data Protection Act (PDPA) incorporating the 2024 Amendments and 2025 Guidelines,NUR INSPIRASI SDN. BHD. via HRD Corp e LATiH,https://elatih.hrdcorp.gov.my/courses-details/malaysia-s-personal-data-protection-act-pdpa-incorporating-the-2024-amendments-and-2025-guidelines,data_privacy;regulatory_compliance,beginner,2.25,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Page lists 2 hour slides plus 15 minute quiz; price and free or premium status not shown on page (e LATiH FAQ says it has both free and premium courses); provider course outline PDF hosted on e LATiH marks it HRDC claimable but uses a slightly different title and 3 hours; level and language not stated, beginner and English (page language) are my judgement Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
A27,Risk management,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/risk-management,risk_management,beginner,24.0,0.0,Free,English,2026-10-24,OpenLearn label Level 1: Introductory; page states 24 hours study over eight sessions; 24 CPD points; free statement of participation and digital badge
A28,Effective communication in the workplace,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/effective-communication-the-workplace/content-section-overview,communication,beginner,24.0,0.0,Free,English,2026-10-24,OpenLearn label Level 1: Introductory; page states 24 hours study; 24 CPD points; free statement of participation and digital badge
A29,Public Speaking and Presentation Skills,Alison,https://alison.com/course/public-speaking-and-presentation-skills,presentation_skills;communication,beginner,4.0,0.0,Free with certificate fee,English,2026-10-24,Page states 3 to 4 hours; the upper figure is used; Alison label Beginner; certificate fee applies (amount not shown)
A30,Introduction to Time Management,Alison,https://alison.com/course/introduction-to-time-management-revised-2017,time_management,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Beginner; certificate fee applies (amount not shown)
A31,Project management: the start of the project journey,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/leadership-management/project-management-the-start-the-project-journey/content-section-0,project_management,advanced,35.0,0.0,Free,English,2026-10-24,"OpenLearn label Level 3: Advanced; page states 35 hours study; covers project origins, planning, project manager role and feasibility studies; free statement of participation"
A32,Lean Business Process Management Change Skills,Alison,https://alison.com/course/lean-business-process-management-change-skills,process_improvement,beginner,4.0,0.0,Free with certificate fee,English,2026-10-24,Page states 3 to 4 hours; the upper figure is used; Alison label Beginner; certificate fee applies (amount not shown)
A33,Stakeholder Relationship Management,Alison,https://alison.com/course/stakeholders-relationships-management,stakeholder_management,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Advanced; CPD accredited; certificate fee applies (amount not shown)
A34,Microsoft Word for Administrative Professionals,Alison,https://alison.com/course/microsoft-word-for-administrative-professionals,microsoft_office;document_management,advanced,4.0,0.0,Free with certificate fee,English,2026-10-24,Page states 3 to 4 hours; the upper figure is used; Alison label Advanced; certificate fee applies (amount not shown)
A35,Document Control: From Principles to Practice,Alison,https://alison.com/course/document-control-from-principles-to-practice,document_management,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,Page states 1.5 to 3 hours; the upper figure is used; Alison label Advanced; CPD accredited; certificate fee applies (amount not shown)
A36,Introduction to Office and Administrative Management,Alison,https://alison.com/course/introduction-to-office-and-administrative-management,communication;time_management;scheduling_coordination,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,"Page states 1.5 to 3 hours; the upper figure is used; Alison label Advanced; covers office administrator duties, business etiquette, workplace organisation and organising meetings; certificate fee applies (amount not shown) Also tagged scheduling_coordination: Description includes organising meetings."
B01,Purchasing and Supplies Management,Alison,https://alison.com/course/purchasing-and-supplies-management,procurement,beginner,4.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 3 to 4 hours; the upper figure is used."
B02,"Effective Practices for Sourcing, Purchasing, and Procurement",Alison,https://alison.com/course/effective-practices-for-sourcing-purchasing-and-procurement,procurement,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B03,Supplier Performance Management (SPM),HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/supplier-performance-management-spm,procurement;risk_management,beginner,,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,Covers the role of suppliers and how to define and assess supplier performance. No trainer or duration shown. Level is my judgement (page lists CEFR Level A2). Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses.
B04,Build Your Powerful Negotiation Skills,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/build-your-powerful-negotiation-skills,vendor_negotiation;contract_management,beginner,,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Trainer: Yoggesvari. Aimed at business negotiators; covers integrative negotiation and making contracts durable with milestones and deadlines. No duration shown. Level is my judgement; page in English, language field not shown. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
B06,Supply Chain Management,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/supply-chain-management,supply_chain_planning,beginner,1.22,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Trainer: Sushma Sadalage. Single curriculum item listed at 1 hour 13 min. Covers supply chain flows, departments and cycles. Level is my judgement (page requires CEFR A2 English). Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
B07,Supply Chain Management and Capacity Planning,Alison,https://alison.com/course/supply-chain-management-and-capacity-planning,supply_chain_planning;production_planning,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Covers capacity planning, forecasting methods, JIT, MRP and lean. Page states 1.5 to 3 hours; the upper figure is used."
B08,Designing a Logistics and Transportation Strategy,Alison,https://alison.com/course/designing-a-logistics-and-transportation-strategy,logistics,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B09,"Warehouse Management: Inventory, Stock and Supply Chains",Alison,https://alison.com/course/warehouse-management-inventory-stock-and-supply-chains,warehouse_management;inventory_management,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B10,A Holistic Approach to Inventory Management,Alison,https://alison.com/course/a-holistic-approach-to-inventory-management,inventory_management,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 3 to 4 hours; the upper figure is used."
B11,Describe warehouse management in Dynamics 365 Supply Chain Management,Microsoft Learn,https://learn.microsoft.com/en-us/training/modules/describe-warehouse-management-dynamics-365/,warehouse_management;erp_systems,beginner,1.68,0.0,Free,English,2026-10-24,"Microsoft Learn FAQ states training is free. Duration of 101 min comes from the page metadata because unit times did not display. Specific to Dynamics 365; 14 units, no prerequisites. Tagged erp_systems because Dynamics 365 Supply Chain Management is an ERP module."
B12,Master planning in Dynamics 365 Supply Chain Management,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/master-planning-supply-chain-management/,production_planning;supply_chain_planning;erp_systems,intermediate,,0.0,Free,English,2026-10-24,"Learning path of 3 modules, including Demand Driven MRP. Microsoft Learn FAQ states training is free. Specific to Dynamics 365. No total duration stated. Tagged erp_systems because Dynamics 365 Supply Chain Management is an ERP module."
B13,Supply chain sustainability,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/leadership-management/supply-chain-sustainability/content-section-0,sustainability;supply_chain_planning,beginner,8.0,0.0,Free,English,2026-10-24,Free course with a free statement of participation. OU Level 1: Introductory.
B14,Certified Lean Six Sigma White Belt Practitioner,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/certified-lean-six-sigma-white-belt-practitioner,quality_management;process_improvement,beginner,,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Trainer: Prabageran Thiagarayah (Duke Training & Consultancy). Introduces Lean Six Sigma and DMAIC; page says beginner friendly. No total duration shown, only quiz times per module. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
B15,The Fundamentals of Project Management,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/the-fundamentals-of-project-management,project_management,beginner,1.2,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,Trainer: Stanicaa. Duration shown as 1 hour 12 min. Level is my judgement (page requires CEFR A2 English). Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses.
B17,Understanding and managing risk,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/understanding-and-managing-risk/content-section-0,risk_management,advanced,20.0,0.0,Free,English,2026-10-24,Free course with a free statement of participation. OU Level 3: Advanced. Includes operational risk management.
B18,[Budgeting Course] Essential Budgeting Hacks for Small Business,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/budgeting-course-essential-budgeting-hacks-for-small-business,fpa_budgeting,beginner,1.25,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,Single curriculum item listed at 1 hour 15 min. Written for small business budgeting and planning. Level is my judgement (page requires CEFR A2 English). Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses.
B19,Corporate Event Planning Essentials,Alison,https://alison.com/course/corporate-event-planning-essentials,event_planning,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 3 to 4 hours; the upper figure is used."
B20,Diploma in Event Planning and Management,Alison,https://alison.com/course/diploma-in-event-planning-and-management,event_planning,advanced,10.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Covers sponsorship, logistics planning, onsite execution, contracts and sustainable planning. Page states 6 to 10 hours; the upper figure is used."
B21,Facilities Management,Alison,https://alison.com/course/facilities-management,facilities_management,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B22,Essentials of Property Management Work,Alison,https://alison.com/course/essentials-of-property-management-work,property_management,beginner,5.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 4 to 5 hours; the upper figure is used."
B23,Fire Safety Management,ILO International Training Centre (ITCILO),https://www.itcilo.org/courses/fire-safety-management,emergency_response;facilities_management;occupational_safety,beginner,,0.0,Free,English,2026-10-24,"Free and self guided on ITCILO eCampus. Two modules: fire safety in buildings, and emergency action plans with templates. Certificate on passing both final tests. No duration stated. Level is my judgement."
B24,Essentials for a Safer and Compliant Workplace - OSHA 1994 and Amendment 2022,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/essentials-for-a-safer-and-compliant-workplace-osha-1994-and-amendment-2022,occupational_safety,beginner,1.0,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Malaysian OSHA 1994 and its 2022 amendment. Description calls it a 1 hour webinar; modules listed at 18, 15 and 23 min. Trainer: Carolene Anthony (KAC Advisory Services PLT). Level and language not stated; level is my judgement and page is in English. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
B25,Workplace Safety for Employees,HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/workplace-safety-for-employees,occupational_safety;hazard_risk_assessment,beginner,0.67,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,Provider: EQUICK LEARN SDN. BHD. Duration 40 min. Covers workplace hazards and employee rights and responsibilities. Level is my judgement. Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses.
B26,Hazard Identification and Risk Assessment (HIRA),Alison,https://alison.com/course/hazard-identification-and-risk-assessment-hira,hazard_risk_assessment;occupational_safety,advanced,30.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 20 to 30 hours; the upper figure is used."
B27,Accident Prevention and Investigation in Occupational Systems,Alison,https://alison.com/course/accident-prevention-and-investigation-in-occupational-systems,incident_investigation;occupational_safety,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B28,ISO 45001:2018 - Principles of Occupational Health and Safety Management Systems,Alison,https://alison.com/course/iso-45001-2018-principles-of-occupational-health-and-safety-management-systems,iso_45001;occupational_safety,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Page states 1.5 to 3 hours; the upper figure is used."
B29,IS-235.C: Emergency Planning,FEMA NDEMU (Emergency Management Institute),https://training.fema.gov/is/courseoverview.aspx?code=is-235.c&lang=en,emergency_response,beginner,5.0,0.0,Free,English,2026-10-24,FEMA NDEMU page states Independent Study courses are free of charge and open to all. The final exam needs a FEMA SID. Based on US emergency planning doctrine; 0.5 CEU. Level is my judgement.
B30,Environmental management and organisations,The Open University (OpenLearn),https://www.open.edu/openlearn/nature-environment/environmental-management-and-organisations/content-section-0,environmental_compliance;sustainability;esg_reporting,intermediate,15.0,0.0,Free,English,2026-10-24,Free course with a free statement of participation. OU Level 2: Intermediate. Includes a section on corporate sustainability reporting.
B31,Environmental Management and Legal Compliance,Alison,https://alison.com/course/environmental-management-and-legal-compliance,environmental_compliance,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Covers industrial pollution, waste control and environmental management systems. Page states 1.5 to 3 hours; the upper figure is used."
B32,Cracking the ESG Opportunity Code (English),HRD Corp e-LATiH,https://elatih.hrdcorp.gov.my/courses-details/cracking-the-esg-opportunity-code-english,esg_reporting;sustainability,beginner,,,e-LATiH (HRD Corp): confirm free or premium after signing in,English,2026-10-24,"Trainer: Cheah Seh Wen (Sin Ma Think Tank Sdn. Bhd.). Covers ESG reporting, its standards and what an ESG report is. Page says beginners welcome. No total duration stated (the listed lessons add up to 75 min). Free status not shown on the course page; the e-LATiH FAQ says the platform offers both free and premium courses."
B33,A Guide to ESG Performance Measurement,Alison,https://alison.com/course/a-guide-to-esg-performance-measurement,esg_reporting;carbon_accounting,beginner,4.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Covers ESG metrics, indicators and categories of greenhouse gas measurement; does not name GRI or ISSB. Page states 3 to 4 hours; the upper figure is used."
B34,ISO 14067 - Managing the Carbon Footprint of Products,Alison,https://alison.com/course/iso-14067-managing-the-carbon-footprint-of-products,carbon_accounting;sustainability,intermediate,5.0,0.0,Free with certificate fee,English,2026-10-24,"Free to study; certificate optional, price not shown on page; CPD accredited. Covers direct and indirect emissions, life cycle inventory and system boundaries. Page states 4 to 5 hours; the upper figure is used."
B35,"Net Zero 101: What, Why and How",UN CC:e-Learn (UNU-IAS),https://unccelearn.org/course/view.php?id=185&page=overview&lang=en,sustainability,beginner,1.5,0.0,Free,English,2026-10-24,"Free and self paced, with a free UNU-IAS certificate (75% needed on each quiz). Covers climate basics and reducing carbon footprint; not a carbon accounting course. Level is my judgement (page calls it foundational)."
S01,Sales and Negotiations Skills,Alison,https://alison.com/courses/sales-and-negotiations-skills/content,b2b_sales;negotiation,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,Alison label Intermediate. Free to study; the CPD accredited certificate is paid but the course page does not show the fee. Page states 1.5 to 3 hours; the upper figure is used. A Spanish version is also listed.
S02,Inbound Strategy and Sales Software by HubSpot Academy,Alison (HubSpot Academy content),https://alison.com/course/inbound-strategy-and-sales-software-by-hubspot-academy,b2b_sales;crm,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Beginner. The course page does not state a price, but the Alison pricing page states unlimited free access to all courses. The certificate is paid and the fee is not shown. Page states 1.5 to 3 hours; the upper figure is used."
S03,Understanding Key Account Management,Alison,https://alison.com/course/understanding-key-account-management,account_management;b2b_sales,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,Alison label Intermediate. Free to study; the certificate fee is not shown on the page. Page states 1.5 to 3 hours; the upper figure is used.
S04,Negotiation Skills for Business Success,Alison,https://alison.com/course/negotiation-skills-for-business-success,negotiation,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,Alison label Intermediate. Free to study; the certificate fee is not shown on the page. Page states 3 to 4 hours; the upper figure is used. A Spanish version is also listed.
S05,Salesforce CRM,Salesforce Trailhead,https://trailhead.salesforce.com/content/learn/modules/lex_implementation_basics,crm,beginner,0.83,0.0,Free,English,2026-10-24,"Trailhead module with 3 units; the page states about 50 minutes. The provider label is Foundational, mapped here to beginner. The page describes it as a free course."
S06,Developing Quality Customer Service Skills,Alison,https://alison.com/course/developing-quality-customer-service-skills,customer_service,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,Alison label Intermediate. Free to study; the certificate fee is not shown on the page. Page states 1.5 to 3 hours; the upper figure is used.
S07,Presentation Skills,Alison,https://alison.com/course/presentation-skills,presentation_skills,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,Alison label Beginner. Free to study; the certificate fee is not shown on the page. Page states 1.5 to 3 hours; the upper figure is used.
S08,IT Support Technician Foundations,Alison,https://alison.com/course/it-support-technician-foundations,it_support;itsm_ticketing,intermediate,4.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Intermediate. Covers service desk operations, logging support requests and prioritising issues by urgency and impact. Free to study; the certificate fee is not shown. Page states 3 to 4 hours; the upper figure is used."
S09,Computer Hardware Basics,Cisco Networking Academy,https://www.netacad.com/courses/computer-hardware-basics?courseLang=en-US,it_support,beginner,,0.0,Free,English,2026-10-24,The page renders as an app. The title and free status come from the page title and description (Free online course). Duration and level are not stated; beginner is my judgement.
S10,Adopt ITSM practices to deliver exceptional service,Atlassian University,https://community.atlassian.com/learning/path/itsm-principles-and-practices,itsm_ticketing,beginner,1.83,0.0,Free,English,2026-10-24,"Learning path of 3 courses, 1 h 50 min. The page does not state a price; the Atlassian blog of 1 May 2024 says all on demand courses are free. No level label; the page targets beginner Jira Service Management users."
S11,Manage customer requests in Jira Service Management,Atlassian University,https://community.atlassian.com/learning/course/manage-customer-requests-in-jira-service-management,itsm_ticketing;customer_service;jira,beginner,0.83,0.0,Free,English,2026-10-24,"4 lessons, 50 min, covering service roles, organising work, customer communication and knowledge base articles. The page does not state a price; Atlassian on demand courses have been free since 1 May 2024. Level is my judgement. Also tagged jira: Taught in Jira Service Management."
S12,Explore Microsoft 365 administration,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/explore-microsoft-365-administration/,windows_m365_admin,beginner,,0.0,Free,English,2026-10-24,"Microsoft Learn label Beginner. Learning path of 3 modules on security foundations, core services and admin controls, and data protection. Duration is not shown on the page."
S13,Stay current with Windows devices and Microsoft 365 Apps,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/stay-current-with-windows/,windows_m365_admin,beginner,,0.0,Free,English,2026-10-24,"Microsoft Learn label Beginner. Learning path of 3 modules on planning and deploying updates for Windows 10, Windows 11 and Microsoft 365 Apps. Duration is not shown on the page."
S14,Networking Basics,Cisco Networking Academy,https://www.netacad.com/courses/networking-basics?courseLang=en-US,networking_basics,beginner,,0.0,Free,English,2026-10-24,The page renders as an app. The title and free status come from the page metadata (Free online course). Duration and level are not stated; beginner is my judgement.
S15,Introduction to Cybersecurity,Cisco Networking Academy,https://www.netacad.com/courses/introduction-to-cybersecurity,cybersecurity_awareness,beginner,,0.0,Free,English,2026-10-24,The page renders as an app. The title and free status come from the page metadata (Free Online Course). It covers protecting personal digital life and the security challenges companies face. Duration and level are not stated; beginner is my judgement.
S16,Fundamentals of cost accounting and environmental management accounting,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/fundamentals-cost-accounting-and-environmental-management-accounting/content-section-0?intro=1,cost_accounting,intermediate,8.0,0.0,Free,English,2026-10-24,"OpenLearn Level 2 Intermediate, 8 hours of study. Covers cost classification and behaviour, fixed and variable costs, and overheads. A free statement of participation is available on completion; sign in required."
S17,Financial statement analysis and interpretation,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/financial-statement-analysis-and-interpretation/content-section-0,financial_reporting,intermediate,12.0,0.0,Free,English,2026-10-24,"OpenLearn Level 2 Intermediate, 12 hours of study. Covers profitability, liquidity, solvency and efficiency ratios with a case study. Free statement of participation."
S18,Fundamentals of accounting,The Open University (OpenLearn),https://www.open.edu/openlearn/money-business/fundamentals-accounting/,accounting;bookkeeping,beginner,8.0,0.0,Free,English,2026-10-24,"OpenLearn Level 1 Introductory, 8 hours of study over four weeks. Covers double entry, the trial balance and the balance sheet. Free statement of participation."
S19,Introduction to bookkeeping and accounting,The Open University (OpenLearn),https://www.open.edu/openlearn/course/view.php?id=1054,bookkeeping;accounting,beginner,8.0,0.0,Free,English,2026-10-24,"OpenLearn Level 1 Introductory, 8 hours of study. A free statement of participation is available on completion; sign in required."
S20,Advanced Pivot Tables in Excel,Alison,https://alison.com/course/advanced-pivot-tables-in-excel,excel_advanced,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison labels it Beginner despite the title. Covers PivotTables, data import and cleaning, grouping, report layouts and custom number formats. The certificate fee is not shown. Page states 1.5 to 3 hours; the upper figure is used."
S21,Agile Requirements Analysis,Alison,https://alison.com/course/agile-requirements-analysis,requirements_gathering;agile_scrum,intermediate,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Intermediate. Covers analysing, prioritising and recording business requirements, user stories, stakeholder analysis and fact finding. The certificate fee is not shown. Page states 1.5 to 3 hours; the upper figure is used."
S22,Business Analysis Fundamentals,Alison,https://alison.com/course/business-analysis-fundamentals,requirements_gathering,beginner,4.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Beginner. The page does not name elicitation; it covers analysis techniques, investigation techniques and stakeholder management. The certificate fee is not shown. Page states 3 to 4 hours; the upper figure is used."
S23,Lean Six Sigma: Yellow Belt,Alison,https://alison.com/course/lean-six-sigma-yellow-belt,process_improvement,intermediate,5.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Intermediate. Covers DMAIC, 5S and spotting process improvement opportunities. The certificate fee is not shown. Page states 4 to 5 hours; the upper figure is used."
S24,An introduction to Agile and Scrum,Alison,https://alison.com/course/an-introduction-to-agile-and-scrum,agile_scrum,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Beginner. Covers user stories, sprint planning, backlog refinement and retrospectives. The certificate fee is not shown. Page states 1.5 to 3 hours; the upper figure is used."
S25,Data analysis: visualisations in Excel,The Open University (OpenLearn),https://www.open.edu/openlearn/science-maths-technology/data-analysis-visualisations-excel/content-section-0?intro=1,data_visualisation;statistics,beginner,6.0,0.0,Free,English,2026-10-24,"OpenLearn Level 1 Introductory, 6 hours of study. Covers frequency tables, histograms, contingency tables and scatter diagrams. Free statement of participation. A separate course from the OpenLearn hypothesis testing course already in the catalogue."
S26,Understanding the Essentials of Excel Data Visualization,Alison,https://alison.com/course/understanding-the-essentials-of-excel-data-visualization,data_visualisation;dashboarding;excel_advanced;kpi_reporting,advanced,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Advanced. Includes building a drilldown dashboard, KPI charts, sparklines and heatmaps. The certificate fee is not shown. Page states 1.5 to 3 hours; the upper figure is used. Also tagged kpi_reporting: Description includes KPI charts and a dashboard."
S27,Query and modify data with Transact-SQL,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/get-started-querying-with-transact-sql/,sql,beginner,,0.0,Free,English,2026-10-24,"Microsoft Learn label Beginner. Learning path of 6 modules covering filtering, joins, subqueries, GROUP BY and modifying data. Duration is not shown on the page."
S28,The Fundamentals of Statistics,Alison,https://alison.com/course/the-fundamentals-of-statistics,statistics,beginner,3.0,0.0,Free with certificate fee,English,2026-10-24,"Alison label Beginner. Covers sampling, central tendency, standard deviation, the normal curve and correlation. The certificate fee is not shown. Page states 1.5 to 3 hours; the upper figure is used."
S29,AI Fluency: Framework & Foundations,Anthropic Academy,https://anthropic.skilljar.com/ai-fluency-framework-foundations,ai_assistants;generative_ai;prompt_engineering,beginner,,0.0,Free,English,2026-10-24,The registration button reads Register FREE. A certificate of completion is awarded after a final assessment. Duration is not stated. The page says it suits all levels; beginner is my judgement.
S30,Build foundational generative AI skills,Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/explore-microsoft-365-copilot-business-chat,ai_assistants;generative_ai;prompt_engineering,beginner,,0.0,Free,English,2026-10-24,"Microsoft Learn label Beginner, aimed at business users. 4 modules, including writing effective prompts and working with Microsoft Copilot Chat. The URL slug does not match the current title. Duration is not shown."
S31,"Draft, analyze, and present with Microsoft Copilot",Microsoft Learn,https://learn.microsoft.com/en-us/training/paths/draft-analyze-present-microsoft-365-copilot/,ai_assistants;generative_ai,intermediate,,0.0,Free,English,2026-10-24,"The English page labels it Intermediate. 7 modules covering documents, presentations, meetings, data insights and email with AI. Duration is not shown on the page."
S32,Introduction to AI,MinnaLearn and University of Helsinki (Elements of AI),https://www.elementsofai.com/,generative_ai,beginner,,0.0,Free,English,2026-10-24,"Part 1 of Elements of AI, a free course for non experts that needs no maths or programming. General AI literacy rather than generative AI tools. Duration is not stated on the page; the FAQ says 4 to 8 hours per part but does not define a part. Beginner is my judgement."
S33,Make life easier with automation,Atlassian University,https://community.atlassian.com/learning/course/make-life-easier-with-automation,workflow_automation;jira,beginner,0.75,0.0,Free,English,2026-10-24,"4 lessons, 45 min, on automating tasks in Jira and Confluence without writing code, including natural language rule building. The page does not state a price; Atlassian on demand courses have been free since 1 May 2024. Level is my judgement. Also tagged jira: Automation in Jira and Confluence."
'''
REFERENCE["sdg_goals.csv"] = r'''
goal,goal_name,colour_hex,official_url
1,No Poverty,#E5243B,https://sdgs.un.org/goals/goal1
2,Zero Hunger,#DDA63A,https://sdgs.un.org/goals/goal2
3,Good Health and Well-being,#4C9F38,https://sdgs.un.org/goals/goal3
4,Quality Education,#C5192D,https://sdgs.un.org/goals/goal4
5,Gender Equality,#FF3A21,https://sdgs.un.org/goals/goal5
6,Clean Water and Sanitation,#26BDE2,https://sdgs.un.org/goals/goal6
7,Affordable and Clean Energy,#FCC30B,https://sdgs.un.org/goals/goal7
8,Decent Work and Economic Growth,#A21942,https://sdgs.un.org/goals/goal8
9,"Industry, Innovation and Infrastructure",#FD6925,https://sdgs.un.org/goals/goal9
10,Reduced Inequalities,#DD1367,https://sdgs.un.org/goals/goal10
11,Sustainable Cities and Communities,#FD9D24,https://sdgs.un.org/goals/goal11
12,Responsible Consumption and Production,#BF8B2E,https://sdgs.un.org/goals/goal12
13,Climate Action,#3F7E44,https://sdgs.un.org/goals/goal13
14,Life Below Water,#0A97D9,https://sdgs.un.org/goals/goal14
15,Life on Land,#56C02B,https://sdgs.un.org/goals/goal15
16,"Peace, Justice and Strong Institutions",#00689D,https://sdgs.un.org/goals/goal16
17,Partnerships for the Goals,#19486A,https://sdgs.un.org/goals/goal17
'''
REFERENCE["sdg_targets.csv"] = r'''
target,goal,target_summary,official_url
1.3,1,"Implement social protection systems and measures for all, including floors.",https://sdgs.un.org/goals/goal1
3.9,3,"Substantially reduce deaths and illnesses from hazardous chemicals and air, water and soil pollution and contamination.",https://sdgs.un.org/goals/goal3
4.3,4,"Equal access for all to affordable, quality technical, vocational and tertiary education.",https://sdgs.un.org/goals/goal4
4.4,4,"Substantially increase the number of youth and adults with relevant skills, including technical and vocational skills, for employment, decent jobs and entrepreneurship.",https://sdgs.un.org/goals/goal4
7.3,7,Double the global rate of improvement in energy efficiency.,https://sdgs.un.org/goals/goal7
8.2,8,"Higher economic productivity through diversification, technological upgrading and innovation.",https://sdgs.un.org/goals/goal8
8.3,8,"Support productive activities, decent job creation and the formalisation and growth of micro, small and medium enterprises.",https://sdgs.un.org/goals/goal8
8.5,8,"Full and productive employment and decent work for all women and men, with equal pay for work of equal value.",https://sdgs.un.org/goals/goal8
8.6,8,"Substantially reduce the proportion of youth not in employment, education or training.",https://sdgs.un.org/goals/goal8
8.7,8,"Take effective measures to eradicate forced labour, modern slavery and human trafficking, and end child labour.",https://sdgs.un.org/goals/goal8
8.8,8,Protect labour rights and promote safe and secure working environments for all workers.,https://sdgs.un.org/goals/goal8
8.9,8,Promote sustainable tourism that creates jobs and promotes local culture and products.,https://sdgs.un.org/goals/goal8
9.5,9,Enhance scientific research and upgrade the technological capabilities of industrial sectors.,https://sdgs.un.org/goals/goal9
9.c,9,Significantly increase access to information and communications technology.,https://sdgs.un.org/goals/goal9
10.2,10,"Empower and promote the social, economic and political inclusion of all, irrespective of age, sex, disability, race, ethnicity, origin, religion or status.",https://sdgs.un.org/goals/goal10
11.6,11,"Reduce the adverse per capita environmental impact of cities, including air quality and waste management.",https://sdgs.un.org/goals/goal11
12.4,12,"Achieve the environmentally sound management of chemicals and all wastes and reduce their release to air, water and soil.",https://sdgs.un.org/goals/goal12
12.5,12,"Substantially reduce waste generation through prevention, reduction, recycling and reuse.",https://sdgs.un.org/goals/goal12
12.6,12,Encourage companies to adopt sustainable practices and integrate sustainability information into their reporting cycle.,https://sdgs.un.org/goals/goal12
12.7,12,Promote sustainable procurement practices (the official target refers to public procurement).,https://sdgs.un.org/goals/goal12
13.3,13,"Improve education, awareness raising and human and institutional capacity on climate change mitigation and adaptation.",https://sdgs.un.org/goals/goal13
16.10,16,Ensure public access to information and protect fundamental freedoms.,https://sdgs.un.org/goals/goal16
16.3,16,Promote the rule of law and ensure equal access to justice for all.,https://sdgs.un.org/goals/goal16
16.5,16,Substantially reduce corruption and bribery in all their forms.,https://sdgs.un.org/goals/goal16
16.6,16,"Develop effective, accountable and transparent institutions at all levels.",https://sdgs.un.org/goals/goal16
17.1,17,"Strengthen domestic resource mobilisation, including capacity for tax and other revenue collection.",https://sdgs.un.org/goals/goal17
17.18,17,"Increase the availability of high quality, timely and reliable data disaggregated by income, gender, age and other characteristics.",https://sdgs.un.org/goals/goal17
17.19,17,Develop measurements of progress on sustainable development and support statistical capacity building.,https://sdgs.un.org/goals/goal17
'''
REFERENCE["sdg_links.csv"] = r'''
entity_type,entity_id,goal,target,strength,rationale,drafted_by,reviewed_by_human
role,paralegal,16,16.3,direct,Accurate legal documents and filings support the rule of law and access to justice.,"Claude (Anthropic), 2026-10-24",no
role,paralegal,16,16.6,supporting,Compliance checks make organisations more accountable.,"Claude (Anthropic), 2026-10-24",no
role,hr_executive,8,8.5,direct,"Fair hiring, pay and policies support decent work for all.","Claude (Anthropic), 2026-10-24",no
role,hr_executive,10,10.2,supporting,"Inclusive HR practice widens opportunity regardless of age, sex or origin.","Claude (Anthropic), 2026-10-24",no
role,ld_trainer,4,4.3,direct,Delivers technical and vocational learning to working adults.,"Claude (Anthropic), 2026-10-24",no
role,ld_trainer,8,8.5,supporting,Reskilling keeps colleagues in productive employment.,"Claude (Anthropic), 2026-10-24",no
role,operations_manager,8,8.2,direct,Better processes and vendor management raise productivity.,"Claude (Anthropic), 2026-10-24",no
role,operations_manager,8,8.5,supporting,Good operations management supports decent working conditions.,"Claude (Anthropic), 2026-10-24",no
role,event_executive,8,8.9,direct,Events and conferences create jobs and promote local culture and products.,"Claude (Anthropic), 2026-10-24",no
role,event_executive,12,12.5,supporting,Event planning choices can cut waste.,"Claude (Anthropic), 2026-10-24",no
role,business_analyst,8,8.2,direct,Redesigns processes and introduces technology to raise productivity.,"Claude (Anthropic), 2026-10-24",no
role,business_analyst,9,9.5,supporting,Helps organisations upgrade their technological capabilities.,"Claude (Anthropic), 2026-10-24",no
role,sales_account_manager,8,8.2,direct,Connects businesses with customers and drives productive growth.,"Claude (Anthropic), 2026-10-24",no
role,sales_account_manager,8,8.3,supporting,Helps small and medium enterprises find customers and grow.,"Claude (Anthropic), 2026-10-24",no
role,it_support,9,9.c,direct,Keeps people connected to and able to use information and communications technology.,"Claude (Anthropic), 2026-10-24",no
role,it_support,4,4.4,supporting,Builds the everyday digital skills of colleagues.,"Claude (Anthropic), 2026-10-24",no
role,data_analyst,8,8.2,direct,"Gives teams evidence for decisions, which raises productivity.","Claude (Anthropic), 2026-10-24",no
role,data_analyst,17,17.18,supporting,"Produces timely, reliable data.","Claude (Anthropic), 2026-10-24",no
role,procurement_buyer,12,12.7,direct,Purchasing choices can favour sustainable suppliers.,"Claude (Anthropic), 2026-10-24",no
role,procurement_buyer,8,8.7,supporting,Supplier checks help keep forced and child labour out of supply chains.,"Claude (Anthropic), 2026-10-24",no
role,recruitment_consultant,8,8.5,direct,"Matches people to productive, decent jobs.","Claude (Anthropic), 2026-10-24",no
role,recruitment_consultant,8,8.6,supporting,Helps young people move into employment.,"Claude (Anthropic), 2026-10-24",no
role,safety_health_officer,8,8.8,direct,Directly promotes safe and secure working environments.,"Claude (Anthropic), 2026-10-24",no
role,safety_health_officer,3,3.9,supporting,Controls exposure to hazardous chemicals and pollution at work.,"Claude (Anthropic), 2026-10-24",no
role,property_facilities,7,7.3,direct,Building operations are a major lever for energy efficiency.,"Claude (Anthropic), 2026-10-24",no
role,property_facilities,11,11.6,supporting,Waste and air quality management in buildings reduce urban impact.,"Claude (Anthropic), 2026-10-24",no
role,senior_accounts,16,16.6,direct,Transparent financial reporting supports accountability.,"Claude (Anthropic), 2026-10-24",no
role,senior_accounts,16,16.5,supporting,Reconciliations and controls help detect errors and fraud.,"Claude (Anthropic), 2026-10-24",no
role,office_supervisor,8,8.5,direct,Leading an admin team well supports decent work.,"Claude (Anthropic), 2026-10-24",no
role,office_supervisor,16,16.6,supporting,Accurate records support accountable organisations.,"Claude (Anthropic), 2026-10-24",no
role,accounts_executive,8,8.5,direct,A highly exposed role; reskilling keeps the worker in productive employment.,"Claude (Anthropic), 2026-10-24",no
role,accounts_executive,16,16.5,supporting,Reconciliations and controls help detect errors and fraud.,"Claude (Anthropic), 2026-10-24",no
role,admin_executive,8,8.5,direct,A highly exposed role; reskilling keeps the worker in productive employment.,"Claude (Anthropic), 2026-10-24",no
role,admin_executive,16,16.6,supporting,Accurate records support accountable organisations.,"Claude (Anthropic), 2026-10-24",no
role,payroll_admin,1,1.3,direct,Correct EPF and SOCSO contributions keep workers in social protection schemes.,"Claude (Anthropic), 2026-10-24",no
role,payroll_admin,8,8.5,supporting,A highly exposed role; reskilling keeps the worker in productive employment.,"Claude (Anthropic), 2026-10-24",no
role,customer_service,8,8.5,direct,An exposed role; reskilling keeps the worker in productive employment.,"Claude (Anthropic), 2026-10-24",no
skill,statistics,17,17.19,direct,Statistical skills are the base of statistical capacity.,"Claude (Anthropic), 2026-10-24",no
skill,statistics,17,17.18,supporting,"Needed to produce reliable, disaggregated data.","Claude (Anthropic), 2026-10-24",no
skill,data_quality,17,17.18,direct,Directly improves the reliability of data.,"Claude (Anthropic), 2026-10-24",no
skill,data_governance,16,16.6,direct,Clear ownership and lineage make data use accountable.,"Claude (Anthropic), 2026-10-24",no
skill,data_governance,17,17.18,supporting,Governed data is more reliable and timely.,"Claude (Anthropic), 2026-10-24",no
skill,data_privacy,16,16.10,direct,Protects personal data and fundamental freedoms.,"Claude (Anthropic), 2026-10-24",no
skill,responsible_ai,10,10.2,direct,"Fairness testing reduces discrimination by age, sex, origin or status.","Claude (Anthropic), 2026-10-24",no
skill,responsible_ai,16,16.6,supporting,Makes AI systems transparent and accountable.,"Claude (Anthropic), 2026-10-24",no
skill,esg_reporting,12,12.6,direct,Integrates sustainability information into company reporting.,"Claude (Anthropic), 2026-10-24",no
skill,carbon_accounting,13,13.3,direct,Builds institutional capacity for climate change mitigation.,"Claude (Anthropic), 2026-10-24",no
skill,carbon_accounting,7,7.3,supporting,Measuring energy use is the first step to improving efficiency.,"Claude (Anthropic), 2026-10-24",no
skill,sustainability,13,13.3,direct,Climate knowledge and awareness.,"Claude (Anthropic), 2026-10-24",no
skill,audit,16,16.5,direct,Audit and internal control help prevent corruption and fraud.,"Claude (Anthropic), 2026-10-24",no
skill,audit,16,16.6,supporting,Makes organisations accountable.,"Claude (Anthropic), 2026-10-24",no
skill,reconciliation,16,16.5,supporting,Reconciliations catch errors and irregular payments.,"Claude (Anthropic), 2026-10-24",no
skill,accounting,16,16.6,supporting,Sound accounts underpin accountable organisations.,"Claude (Anthropic), 2026-10-24",no
skill,financial_reporting,16,16.6,direct,Transparent financial reporting supports accountability.,"Claude (Anthropic), 2026-10-24",no
skill,tax,17,17.1,direct,Supports domestic tax and revenue collection.,"Claude (Anthropic), 2026-10-24",no
skill,einvoicing,17,17.1,direct,E invoicing improves tax compliance and revenue collection.,"Claude (Anthropic), 2026-10-24",no
skill,einvoicing,8,8.3,supporting,Helps small businesses formalise and digitalise.,"Claude (Anthropic), 2026-10-24",no
skill,payroll,1,1.3,direct,Correct EPF and SOCSO contributions keep workers in social protection schemes.,"Claude (Anthropic), 2026-10-24",no
skill,payroll,8,8.5,supporting,"Accurate, timely pay is part of decent work.","Claude (Anthropic), 2026-10-24",no
skill,procurement,12,12.7,supporting,Procurement choices can favour sustainable suppliers.,"Claude (Anthropic), 2026-10-24",no
skill,workflow_automation,8,8.2,direct,Automation raises productivity.,"Claude (Anthropic), 2026-10-24",no
skill,rpa,8,8.2,direct,Automation raises productivity.,"Claude (Anthropic), 2026-10-24",no
skill,power_automate,8,8.2,direct,Automation raises productivity.,"Claude (Anthropic), 2026-10-24",no
skill,ai_assistants,8,8.2,direct,Using AI tools well raises productivity at work.,"Claude (Anthropic), 2026-10-24",no
skill,generative_ai,8,8.2,direct,A general purpose technology for productivity.,"Claude (Anthropic), 2026-10-24",no
skill,generative_ai,9,9.5,supporting,Upgrades technological capability.,"Claude (Anthropic), 2026-10-24",no
skill,llm,9,9.5,supporting,Upgrades technological capability.,"Claude (Anthropic), 2026-10-24",no
skill,machine_learning,9,9.5,direct,Applies research methods to industry.,"Claude (Anthropic), 2026-10-24",no
skill,deep_learning,9,9.5,direct,Applies research methods to industry.,"Claude (Anthropic), 2026-10-24",no
skill,kpi_reporting,16,16.6,supporting,Clear performance metrics support accountability.,"Claude (Anthropic), 2026-10-24",no
skill,occupational_safety,8,8.8,direct,Safety and health work protects workers directly.,"Claude (Anthropic), 2026-10-24",no
skill,hazard_risk_assessment,8,8.8,direct,Finding and controlling hazards prevents injuries at work.,"Claude (Anthropic), 2026-10-24",no
skill,incident_investigation,8,8.8,supporting,Learning from incidents prevents the next one.,"Claude (Anthropic), 2026-10-24",no
skill,iso_45001,8,8.8,direct,A safety management system makes workplaces safer by design.,"Claude (Anthropic), 2026-10-24",no
skill,emergency_response,8,8.8,supporting,Preparedness protects workers in an emergency.,"Claude (Anthropic), 2026-10-24",no
skill,environmental_compliance,12,12.4,direct,Sound handling of chemicals and scheduled waste reduces pollution.,"Claude (Anthropic), 2026-10-24",no
skill,environmental_compliance,11,11.6,supporting,Waste and emissions control reduces urban environmental impact.,"Claude (Anthropic), 2026-10-24",no
skill,employment_law,8,8.8,direct,Knowing employment law protects labour rights.,"Claude (Anthropic), 2026-10-24",no
skill,employee_relations,8,8.8,supporting,Fair grievance and disciplinary processes protect workers.,"Claude (Anthropic), 2026-10-24",no
skill,compensation_benefits,8,8.5,supporting,Fair pay structures support equal pay for work of equal value.,"Claude (Anthropic), 2026-10-24",no
skill,recruitment,8,8.5,supporting,"Fair, open hiring widens access to decent jobs.","Claude (Anthropic), 2026-10-24",no
skill,training_delivery,4,4.3,direct,Delivers vocational learning to adults.,"Claude (Anthropic), 2026-10-24",no
skill,instructional_design,4,4.3,supporting,Well designed courses make vocational learning accessible.,"Claude (Anthropic), 2026-10-24",no
skill,people_management,8,8.5,supporting,Good managers make work decent and productive.,"Claude (Anthropic), 2026-10-24",no
skill,legal_research,16,16.3,supporting,Sound legal research supports access to justice.,"Claude (Anthropic), 2026-10-24",no
skill,legal_documentation,16,16.3,supporting,Accurate legal documents support the rule of law.,"Claude (Anthropic), 2026-10-24",no
skill,regulatory_compliance,16,16.5,direct,Anti money laundering and compliance checks fight corruption and illicit flows.,"Claude (Anthropic), 2026-10-24",no
skill,corporate_secretarial,16,16.6,supporting,Statutory filings keep companies transparent and accountable.,"Claude (Anthropic), 2026-10-24",no
skill,vendor_negotiation,12,12.7,supporting,Supplier selection can favour sustainable practices.,"Claude (Anthropic), 2026-10-24",no
skill,inventory_management,12,12.5,supporting,Good stock control cuts overstock and waste.,"Claude (Anthropic), 2026-10-24",no
skill,supply_chain_planning,12,12.5,supporting,Planning demand well reduces waste across the supply chain.,"Claude (Anthropic), 2026-10-24",no
skill,facilities_management,7,7.3,direct,Building maintenance and controls drive energy efficiency.,"Claude (Anthropic), 2026-10-24",no
skill,property_management,11,11.6,supporting,Building and tenant management shapes waste and air quality.,"Claude (Anthropic), 2026-10-24",no
skill,event_planning,8,8.9,supporting,"Events support local jobs, culture and products.","Claude (Anthropic), 2026-10-24",no
skill,event_planning,12,12.5,supporting,Event planning choices can cut waste.,"Claude (Anthropic), 2026-10-24",no
skill,it_support,9,9.c,direct,Keeps people able to use information and communications technology.,"Claude (Anthropic), 2026-10-24",no
skill,networking_basics,9,9.c,supporting,Reliable networks widen access to ICT.,"Claude (Anthropic), 2026-10-24",no
skill,cybersecurity_awareness,9,9.c,supporting,Safe use of ICT protects people and organisations.,"Claude (Anthropic), 2026-10-24",no
skill,b2b_sales,8,8.3,supporting,Helps enterprises find customers and grow.,"Claude (Anthropic), 2026-10-24",no
skill,cost_accounting,8,8.2,supporting,Knowing true costs helps organisations become more productive.,"Claude (Anthropic), 2026-10-24",no
'''
REFERENCE["sources.csv"] = r'''
source_id,name,publisher,url,type,terms_or_notes,retrieved_on,used_for
hn_whoishiring,Hacker News 'Ask HN: Who is hiring?' monthly threads,"Y Combinator (Hacker News), via the public HN Algolia API",https://hn.algolia.com/api,dynamic,Public job posts. Raw text stays out of the repo; only aggregates and links to each post are committed.,See data/raw/hn/_retrieval_log.json and summary.json,"Skill trends, AI Adoption Index, role skill profiles, citations"
linkedin_kaggle,"LinkedIn Job Postings (2023 to 2024), file postings.csv","Kaggle, dataset arshkon/linkedin-job-postings",https://www.kaggle.com/datasets/arshkon/linkedin-job-postings,static,Postings listed 24 March to 20 April 2024. Not redistributed; downloaded by each user from Kaggle under its licence.,2026-10-22,Role skill profiles only (not trends)
ilo_wp140,"Generative AI and Jobs: A Refined Global Index of Occupational Exposure (ILO Working Paper 140, Gmyrek et al., 2025)",International Labour Organization,https://webapps.ilo.org/static/english/intserv/working-papers/wp140/index.html,static,"Copyright ILO 2025. All 427 occupations from Table A1 (pp. 48 to 61) with page numbers; gradient definitions in Table 5, p. 38.",2026-10-22,GenAI exposure of current and target occupations
isco08,International Standard Classification of Occupations (ISCO-08),International Labour Organization,https://www.ilo.org/public/english/bureau/stat/isco/isco08/index.htm,static,Reference classification,2026-10-22,Linking roles to ILO exposure scores
un_sdg,Sustainable Development Goals and targets,United Nations,https://sdgs.un.org/goals,static,Reference,2026-10-22,"SDG goal names, colours and target summaries"
talentcorp_impact,"Impact Study of AI, Digital and Green Economy on the Malaysian Workforce",TalentCorp Malaysia,https://www.mymahir.my,static,Background context only; no figures used in scoring,2026-10-22,Problem framing
course_providers,Course pages of each provider listed in courses.csv (core and extra courses),Various (see courses.csv url column),see courses.csv,"static, rechecked manually",Each row checked on the date in checked_on (22 or 24 October 2026). e-LATiH rows: free or premium status to confirm after signing in.,see courses.csv,Reskilling pathway
synthetic_cvs,Synthetic sample CVs,"Team, drafted with Claude (Anthropic) and reviewed by the team",data/cvs,synthetic,No real personal data,2026-10-22,Demo profiles and fairness test
team_taxonomy,"Skill taxonomy, role patterns, AI terms and SDG mapping","Team, drafted with Claude (Anthropic); informed by ESCO and O*NET skill groupings",data/reference,curated,Human review status recorded per row where relevant,2026-10-24,"Skill extraction, role matching, SDG links"
'''


version_file = REF / "_version.txt"
old_version = version_file.read_text().strip() if version_file.exists() else None
if old_version != REFERENCE_VERSION and any(REF.glob("*.csv")):
    backup = ROOT / "data" / f"reference_backup_{datetime.now():%Y%m%d_%H%M%S}"
    shutil.move(str(REF), str(backup))
    REF.mkdir(parents=True)
    print(f"Reference version changed ({old_version} to {REFERENCE_VERSION}); old tables moved to {backup.name}")

for name, text in REFERENCE.items():
    path = REF / name
    if OVERWRITE_REFERENCE or not path.exists():
        path.write_text(text.lstrip("\n"), encoding="utf-8")
        print(f"{name:<20} written")
    else:
        print(f"{name:<20} kept (already exists)")
version_file.write_text(REFERENCE_VERSION)

skills.csv           kept (already exists)
roles.csv            kept (already exists)
ai_terms.csv         kept (already exists)
ilo_exposure.csv     kept (already exists)
courses.csv          kept (already exists)
sdg_goals.csv        kept (already exists)
sdg_targets.csv      kept (already exists)
sdg_links.csv        kept (already exists)
sources.csv          kept (already exists)


12

In [3]:
# Synthetic CVs for the demo

CV_SAMPLES = {}
CV_SAMPLES["cv01_accounts_executive.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo.

Name: Nurul Izzati binti Kamal
Location: Shah Alam, Selangor
Current role: Accounts Executive, 5 years

SUMMARY
Accounts executive at a mid sized distribution company. I handle accounts payable and receivable for about 300 suppliers and customers. Our finance team started using automated invoice capture in 2025, and much of my keying work is now done by the system. I want to move into a role where I build reports instead of typing entries.

EXPERIENCE
Accounts Executive, Lim Brothers Distribution Sdn Bhd, Shah Alam (Mar 2021 to present)
* Process supplier invoices and customer billing in AutoCount and SQL Account
* Perform monthly bank reconciliations and supplier statement reconciliations
* Prepare month end closing schedules and accruals for the finance manager
* Submit sales invoices through the MyInvois portal and track SST
* Build Excel workbooks with pivot tables and VLOOKUP to track overdue debtors
* Negotiated payment terms with suppliers and chased overdue customer accounts

Accounts Assistant, Seri Murni Trading, Klang (Jan 2019 to Feb 2021)
* Data entry of purchase orders and payment vouchers
* Filing and document management for the annual audit

EDUCATION
Diploma in Accountancy

SKILLS
AutoCount, SQL Account, Excel (pivot tables, VLOOKUP), reconciliation, accounts payable, accounts receivable, data entry, Microsoft Office

INTERESTS
Helping small businesses go digital; turning numbers into clear charts
'''
CV_SAMPLES["cv02_admin_executive.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Tan Mei Ling
Location: Bayan Lepas, Penang
Current role: Admin Executive, 8 years

SUMMARY
Admin executive at an electronics manufacturing plant. I run office administration, purchasing paperwork and records for a team of 120. The company is rolling out an AI assistant for document handling, and the data entry, filing and scheduling parts of my job are shrinking. I enjoy organising information and fixing messy processes.

EXPERIENCE
Admin Executive, Penang Precision Components Sdn Bhd (Jun 2018 to present)
* Data entry of purchase orders and delivery orders into SAP
* Calendar management for two directors and travel arrangements for visiting engineers
* Records management and document control for ISO certification
* Built a Google Sheets tracker fed by Google Forms for stationery and asset requests
* Coordinate quotations with vendors for procurement
* Event coordination for the annual dinner and two supplier days (300 guests)
* Organise fire drills and keep first aid kits and the emergency contact list up to date
* Supervising staff: two admin clerks

Admin Assistant, Northern Haulage Sdn Bhd, Butterworth (2016 to 2018)
* Customer service by phone and email
* Kept the filing system for shipment documents

EDUCATION
SPM; Certificate in Office Management

SKILLS
Microsoft Office, Google Sheets, data entry, document management, procurement, customer service, SAP (user level)

INTERESTS
Making processes faster; environmental volunteering (beach clean ups)
'''
CV_SAMPLES["cv03_finance_executive.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Arvind a/l Selvam
Location: Petaling Jaya, Selangor
Current role: Finance Executive, 6 years

SUMMARY
Finance executive in a listed property group. I prepare management accounts, budgets and variance analysis, and I already build basic Power BI dashboards. Since 2025 I also collect ESG data for the group sustainability report. I want a role where analysis is my main job rather than recording transactions.

EXPERIENCE
Finance Executive, Menara Harta Bhd, Kuala Lumpur (Jan 2020 to present)
* Prepare monthly management accounts and financial reporting packs
* Lead annual budgeting and quarterly variance analysis for 12 projects
* Use Power Query in Excel to combine SAP exports; built 4 Power BI dashboards for project KPIs
* Collect energy and waste data from sites for the ESG section of the sustainability report
* Prepare schedules for external auditors
* Cost analysis of project budgets with site managers

Audit Associate, Tiara Audit and Advisory, Kuala Lumpur (2018 to 2019)
* Statutory audit fieldwork for SMEs under MFRS

EDUCATION
Bachelor of Accounting (Hons)

SKILLS
Excel (Power Query, pivot tables), Power BI (basic DAX), SAP, budgeting, variance analysis, financial reporting, MFRS, ESG data collection

INTERESTS
Green buildings and climate; mentoring junior staff
'''
CV_SAMPLES["cv04_fresh_graduate.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Lee Jia Hui
Location: Kajang, Selangor
Current status: Fresh graduate, Bachelor of Accounting and Finance

SUMMARY
Fresh accounting graduate. My internship showed me how much junior accounting work is now automated. I completed a short Python course, enjoyed it, and want to start my career in data analytics rather than audit.

EXPERIENCE
Audit Intern, Sterling Assurance PLT, Kuala Lumpur (Nov 2025 to Feb 2026)
* Vouching and bank reconciliation testing for audit clients
* Used Excel to sample transactions and summarise findings

Part time Tutor, home tuition (2023 to 2025)
* Taught SPM mathematics, including probability and statistics basics

EDUCATION
Bachelor of Accounting and Finance (2026). Final year project: regression analysis of SME loan defaults in Excel

COURSES
Kaggle Learn: Python (completed 2026)

SKILLS
Excel, Python (beginner), statistics, regression, reconciliation, communication skills

INTERESTS
Sustainable finance; volunteering with a financial literacy NGO
'''
CV_SAMPLES["cv05_customer_ops.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Muhammad Hafiz bin Zulkifli
Location: Cyberjaya, Selangor
Current role: Customer Operations Coordinator, 4 years

SUMMARY
Coordinator in a shared services centre that handles billing queries for an insurance client. Our team now uses an AI chatbot for first line replies, so manual handling is shrinking. I have become the person who automates things: I use ChatGPT to draft templates and Zapier to move data between tools.

EXPERIENCE
Customer Operations Coordinator, Cyber Shared Services Sdn Bhd (Feb 2022 to present)
* Handle escalated customer service tickets in Zendesk
* Produce weekly KPI reports on response times and backlog in Excel and Google Sheets
* Built Zapier automations that create tickets from emails and update a tracker
* Wrote prompt templates in ChatGPT for agents, using careful prompting to cut drafting time
* Run process improvement sessions with team leads
* Delivering training workshops for new agents
* First point of contact for troubleshooting laptop and printer issues on the floor

Customer Service Executive, Telco Contact Centre, Kuala Lumpur (2020 to 2022)
* Answered billing and account queries by phone and chat

EDUCATION
Diploma in Business Administration

SKILLS
Zendesk, Excel, Google Sheets, KPI reporting, Zapier, ChatGPT, prompting, process improvement, customer service

INTERESTS
New AI tools; teaching colleagues shortcuts
'''
CV_SAMPLES["cv06_payroll_admin.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Rosnah binti Ahmad
Location: Johor Bahru, Johor
Current role: Payroll and HR Administrator, 10 years

SUMMARY
Payroll administrator for a manufacturing company with 450 staff. The company is moving payroll to a cloud HR system with automatic statutory calculations, so my manual payroll work will shrink. I know our employee data very well and care about keeping it accurate and private.

EXPERIENCE
Payroll and HR Administrator, Johor Fabrication Sdn Bhd (2016 to present)
* Monthly payroll processing, including EPF, SOCSO and PCB submissions
* Maintain employee records in the HRIS and keep personal data compliant with PDPA
* Data entry of overtime claims and leave records
* Prepare monthly headcount reports in Excel for management
* New hire onboarding paperwork and induction programmes for factory staff
* Advise staff on leave entitlement under the Employment Act
* Support the HR manager on grievances and disciplinary hearings

Clerk, Johor Bahru district office (2012 to 2016)
* Records management and filing

EDUCATION
SPM; Diploma in Human Resource Management

SKILLS
Payroll, EPF and SOCSO, HRIS, PDPA compliance, Excel (basic), data entry, records management

INTERESTS
Community work; helping colleagues with forms
'''
CV_SAMPLES["cv07a_twin.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Siti Hajar binti Osman
Gender: Female
Age: 52
Location: Ipoh, Perak
Current role: Accounts Assistant, 6 years

SUMMARY
Accounts assistant at an accounting firm serving small businesses. Bank feeds and automatic matching in Xero now do most of the reconciliation work I used to do by hand. I have started learning SQL and Power BI in my own time and want to move into a reporting or data role.

EXPERIENCE
Accounts Assistant, Perak Business Advisory, Ipoh (2020 to present)
* Bookkeeping for 40 SME clients in Xero and QuickBooks
* Monthly bank reconciliations and accounts receivable follow up
* Excel reports with pivot tables for client management meetings
* Shared Google Sheets trackers for client documents

EDUCATION
Diploma in Accounting

COURSES
Kaggle Learn: Intro to SQL (completed 2026)
Microsoft Learn: Power BI beginner modules (in progress)

SKILLS
Xero, QuickBooks, bookkeeping, reconciliation, Excel (pivot tables), Google Sheets, SQL (beginner), Power BI (beginner)

INTERESTS
Small business growth; reading
'''
CV_SAMPLES["cv07b_twin.txt"] = r'''
SYNTHETIC SAMPLE CV, created for the SDC Hackathon demo. Not a real person; employers are fictional.

Name: Daniel Lim Wei Jie
Gender: Male
Age: 29
Location: Ipoh, Perak
Current role: Accounts Assistant, 6 years

SUMMARY
Accounts assistant at an accounting firm serving small businesses. Bank feeds and automatic matching in Xero now do most of the reconciliation work I used to do by hand. I have started learning SQL and Power BI in my own time and want to move into a reporting or data role.

EXPERIENCE
Accounts Assistant, Perak Business Advisory, Ipoh (2020 to present)
* Bookkeeping for 40 SME clients in Xero and QuickBooks
* Monthly bank reconciliations and accounts receivable follow up
* Excel reports with pivot tables for client management meetings
* Shared Google Sheets trackers for client documents

EDUCATION
Diploma in Accounting

COURSES
Kaggle Learn: Intro to SQL (completed 2026)
Microsoft Learn: Power BI beginner modules (in progress)

SKILLS
Xero, QuickBooks, bookkeeping, reconciliation, Excel (pivot tables), Google Sheets, SQL (beginner), Power BI (beginner)

INTERESTS
Small business growth; reading
'''


cv_version = CVS / "_version.txt"
refresh_cvs = not cv_version.exists() or cv_version.read_text().strip() != REFERENCE_VERSION
for name, text in CV_SAMPLES.items():
    path = CVS / name
    if refresh_cvs or not path.exists():
        path.write_text(text.lstrip("\n"), encoding="utf-8")
cv_version.write_text(REFERENCE_VERSION)
print(sorted(p.name for p in CVS.glob("cv*.txt")))

['cv01_accounts_executive.txt', 'cv02_admin_executive.txt', 'cv03_finance_executive.txt', 'cv04_fresh_graduate.txt', 'cv05_customer_ops.txt', 'cv06_payroll_admin.txt', 'cv07a_twin.txt', 'cv07b_twin.txt']


In [4]:
# Load the reference tables

skills = pd.read_csv(REF / "skills.csv")
roles = pd.read_csv(REF / "roles.csv", dtype={"isco_code": str})
ai_terms = pd.read_csv(REF / "ai_terms.csv")
ilo = pd.read_csv(REF / "ilo_exposure.csv", dtype={"isco_code": str})
courses = pd.read_csv(REF / "courses.csv")
sdg_goals = pd.read_csv(REF / "sdg_goals.csv")
sdg_targets = pd.read_csv(REF / "sdg_targets.csv", dtype={"target": str})
sdg_links = pd.read_csv(REF / "sdg_links.csv", dtype={"target": str})

SKILLS = skills["skill_id"].tolist()
SKILL_NAME = skills.set_index("skill_id")["skill_name"]
SKILL_RE = {r.skill_id: re.compile(r.pattern, re.IGNORECASE) for r in skills.itertuples()}
ROLE_RE = {r.role_id: re.compile(r.title_pattern, re.IGNORECASE) for r in roles.itertuples()}
AI_RE = [re.compile(p, re.IGNORECASE) for p in ai_terms["pattern"]]
ROLE_NAME = roles.set_index("role_id")["role_name"]
GOAL_NAME = sdg_goals.set_index("goal")["goal_name"]
ILO = ilo.set_index("isco_code")
EXPOSURE_RANK = {"Not Exposed": 0, "Minimal Exposure": 1, "Gradient 1": 2, "Gradient 2": 3, "Gradient 3": 4, "Gradient 4": 5}
roles["ilo_gradient"] = roles["isco_code"].map(ILO["gradient"])
roles["ilo_score"] = roles["isco_code"].map(ILO["mean_score"])

# Soft skills are shown separately since those appear in most job posts but are rarely written in CVs

TRANSFERABLE = set(skills.loc[skills["category"] == "soft", "skill_id"]) | {"stakeholder_management"}

courses["skills_list"] = courses["skills"].str.split(";")
LEVEL_RANK = {"beginner": 0, "intermediate": 1, "advanced": 2}
courses["level_rank"] = courses["level"].map(LEVEL_RANK).fillna(1)

baseline = pd.DataFrame({"entity_type": "skill", "entity_id": SKILLS, "goal": 4, "target": "4.4",
                         "strength": "baseline",
                         "rationale": "Learning this skill is itself progress on relevant skills for decent jobs.",
                         "reviewed_by_human": "no"})
sdg_map = pd.concat([sdg_links.drop(columns=["drafted_by"], errors="ignore"), baseline], ignore_index=True)

assert not set(courses["skills_list"].explode()) - set(SKILLS), "a course lists an unknown skill"
assert courses["course_id"].is_unique, "duplicate course id"
assert set(roles["role_id"]) <= set(sdg_links.loc[sdg_links["entity_type"] == "role", "entity_id"]), "a role has no SDG link"
assert roles["ilo_gradient"].notna().all(), "a role has no ILO exposure row"
print(f"{len(SKILLS)} skills, {len(roles)} roles, {len(ai_terms)} AI terms, {len(courses)} courses, "
      f"{len(sdg_map)} SDG links, {len(ilo)} ILO occupations")
roles[["role_id", "role_type", "isco_code", "isco_title", "ilo_gradient", "ilo_score"]]

128 skills, 19 roles, 12 AI terms, 127 courses, 228 SDG links, 427 ILO occupations


,role_id,role_type,isco_code,isco_title,ilo_gradient,ilo_score
0,paralegal,target,3411,Legal and Related Associate Professionals,Gradient 1,0.39
1,hr_executive,target,2423,Personnel and Careers Professionals,Gradient 2,0.45
2,ld_trainer,target,2424,Training and Staff Development Professionals,Gradient 2,0.41
3,operations_manager,target,1219,Business Services and Administration Managers Not Elsewhere Classified,Gradient 2,0.42
4,event_executive,target,3332,Conference and Event Planners,Gradient 2,0.48
5,business_analyst,target,2421,Management and Organization Analysts,Gradient 2,0.46
6,sales_account_manager,target,3322,Commercial Sales Representatives,Gradient 2,0.49
7,it_support,target,3512,Information and Communications Technology User Support Technicians,Gradient 2,0.47
8,data_analyst,target,2511,Systems Analysts,Gradient 2,0.49
9,procurement_buyer,target,3323,Buyers,Minimal Exposure,0.39
